In [1]:
import os
# os.environ["CUDA_LAUNCH_BLOCKING"]="1"
# os.environ["TRITON_INTERPRET"] = "1"
os.environ["TORCHDYNAMO_VERBOSE"]="1"

In [2]:
import torch, triton
print("torch", torch.__version__, "| triton", triton.__version__)
print("cuda available:", torch.cuda.is_available())
!nvidia-smi -L

torch 2.9.1+cu129 | triton 3.5.1
cuda available: True
GPU 0: NVIDIA RTX A6000 (UUID: GPU-9ea2a5bb-9945-7a39-de98-ee442fda6d52)
GPU 1: NVIDIA RTX A6000 (UUID: GPU-8a41186c-ba13-9a45-155d-f7d11f840dd5)
GPU 2: NVIDIA RTX A6000 (UUID: GPU-44cf39c0-6589-f9b4-c2e9-9f9fe56949cc)
GPU 3: NVIDIA RTX A6000 (UUID: GPU-d07a0da1-c504-7afc-c543-e21eee74e0c9)
GPU 4: NVIDIA RTX A6000 (UUID: GPU-177d11f7-8350-9199-2059-20d404f8b21a)
GPU 5: NVIDIA RTX A6000 (UUID: GPU-bf34cc97-16d5-ad30-ce20-ee27d60a781d)
GPU 6: NVIDIA RTX A6000 (UUID: GPU-b28a9204-b2ca-f63e-eeb0-43ca5336f76c)
GPU 7: NVIDIA RTX A6000 (UUID: GPU-95a16f6d-1cc3-3d6f-481c-50e26078c0d5)


In [3]:
torch.cuda.set_device(2)      # or "cuda:2"
# now the current device is cuda:2 for everything that follows

In [4]:
# !ls /usr/include/python3.12/Python.h 2>/dev/null && echo "Python.h OK" || echo "MISSING Python.h"
# !find / -name "libcuda.so*" 2>/dev/null

In [5]:
import torch
import triton
import triton.language as tl
import torch.nn.functional as F

In [6]:
DEVICE = torch.device("cuda:2" )
DEVICE

device(type='cuda', index=2)

In [7]:
#Time to build flex attention
#I will pass ptrs of mask.
from torch.nn.attention.flex_attention import flex_attention, create_block_mask

DEVICE = "cuda:2"
DTYPE = torch.bfloat16
HEAD_DIM = 128
NUM_HEADS = 12
NUM_DOCS = 8
SEQ_LEN = 1024

In [8]:
import torch
import triton
import triton.language as tl
from torch.nn.attention import sdpa_kernel, SDPBackend
import torch.nn.functional as F

DEVICE = "cuda:2"
DTYPE = torch.bfloat16

# -----------------------------------------------------------
# 1. KERNEL 1: Compute dQ (Anchors Q, loops over K/V)
# -----------------------------------------------------------
@triton.autotune(
    configs=[
        # A6000 strict SRAM limit configs (< 99KB)
        # 1. Balanced blocks, low pipeline
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 64}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 32}, num_warps=4, num_stages=2),
        
        # 2. Maximum square block, NO pipelining (1 stage only)
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 64}, num_warps=8, num_stages=1), 
        
        # 3. Small square, deeper pipeline for latency hiding
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 32}, num_warps=4, num_stages=3),
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 32}, num_warps=8, num_stages=3),
        
        # 4. Extreme skew for edge cases
        triton.Config({'BLOCK_M': 16, 'BLOCK_N': 64}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 16}, num_warps=4, num_stages=2),
    ],
    key=['seq_len', 'HEAD_DIM'],
)
@triton.jit
def bwd_kernel_dq(
    q_ptr, k_ptr, v_ptr, do_ptr, dq_ptr, lse_ptr, odo_sum_ptr,
    seq_len: tl.constexpr, sm_scale: tl.constexpr, HEAD_DIM: tl.constexpr, 
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr
):
    pid_m = tl.program_id(0) # Q block index
    pid_s = tl.program_id(1)
    pid_h = tl.program_id(2)

    stride_tok = tl.num_programs(2) * HEAD_DIM 
    stride_h = tl.num_programs(1) * seq_len
    base_offset = pid_s * stride_tok * seq_len + pid_h * HEAD_DIM

    # Anchor Q, dO, dQ
    q_block = tl.make_block_ptr(base=q_ptr + base_offset, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(pid_m * BLOCK_M, 0), block_shape=(BLOCK_M, HEAD_DIM), order=(1, 0))
    do_block = tl.make_block_ptr(base=do_ptr + base_offset, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(pid_m * BLOCK_M, 0), block_shape=(BLOCK_M, HEAD_DIM), order=(1, 0))
    dq_block = tl.make_block_ptr(base=dq_ptr + base_offset, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(pid_m * BLOCK_M, 0), block_shape=(BLOCK_M, HEAD_DIM), order=(1, 0))
    
    lse_block = tl.make_block_ptr(base=lse_ptr + pid_h * stride_h + pid_s * seq_len, shape=(seq_len,), strides=(1,), offsets=(pid_m * BLOCK_M,), block_shape=(BLOCK_M,), order=(0,))
    odo_block = tl.make_block_ptr(base=odo_sum_ptr + pid_h * stride_h + pid_s * seq_len, shape=(seq_len,), strides=(1,), offsets=(pid_m * BLOCK_M,), block_shape=(BLOCK_M,), order=(0,))

    # Sliding K and V
    k_block = tl.make_block_ptr(base=k_ptr + base_offset, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(0, 0), block_shape=(BLOCK_N, HEAD_DIM), order=(1, 0))
    v_block = tl.make_block_ptr(base=v_ptr + base_offset, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(0, 0), block_shape=(BLOCK_N, HEAD_DIM), order=(1, 0))

    q = tl.load(q_block, boundary_check=(0,), padding_option="zero")
    do = tl.load(do_block, boundary_check=(0,), padding_option="zero")
    lse = tl.load(lse_block, boundary_check=(0,), padding_option="zero")
    odo = tl.load(odo_block, boundary_check=(0,), padding_option="zero")

    acc_dq = tl.zeros((BLOCK_M, HEAD_DIM), tl.float32)
    qk_scale = sm_scale * 1.44269504

    for n in range(0, seq_len, BLOCK_N):
        k = tl.load(k_block, boundary_check=(0,), padding_option="zero")
        v = tl.load(v_block, boundary_check=(0,), padding_option="zero")

        s = tl.dot(q, tl.trans(k)) * qk_scale
        p = tl.math.exp2(s - lse[:, None])
        
        dp = tl.dot(do, tl.trans(v), out_dtype=tl.float32)
        ds = p * (dp - odo[:, None])
        
        acc_dq += tl.dot(ds.to(tl.bfloat16), k, out_dtype=tl.float32)

        k_block = tl.advance(k_block, (BLOCK_N, 0))
        v_block = tl.advance(v_block, (BLOCK_N, 0))

    acc_dq = acc_dq * sm_scale
    tl.store(dq_block, acc_dq.to(tl.bfloat16), boundary_check=(0,))

# -----------------------------------------------------------
# 2. KERNEL 2: Compute dK, dV (Anchors K/V, loops over Q)
# -----------------------------------------------------------
@triton.autotune(
    configs=[
        # A6000 strict SRAM limit configs (< 99KB)
        # 1. Balanced blocks, low pipeline
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 64}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 32}, num_warps=4, num_stages=2),
        
        # 2. Maximum square block, NO pipelining (1 stage only)
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 64}, num_warps=8, num_stages=1), 
        
        # 3. Small square, deeper pipeline for latency hiding
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 32}, num_warps=4, num_stages=3),
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 32}, num_warps=8, num_stages=3),
        
        # 4. Extreme skew for edge cases
        triton.Config({'BLOCK_M': 16, 'BLOCK_N': 64}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 16}, num_warps=4, num_stages=2),
    ],
    key=['seq_len', 'HEAD_DIM'],
)
@triton.jit
def bwd_kernel_dk_dv(
    q_ptr, k_ptr, v_ptr, do_ptr, dk_ptr, dv_ptr, lse_ptr, odo_sum_ptr,
    seq_len: tl.constexpr, sm_scale: tl.constexpr, HEAD_DIM: tl.constexpr, 
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr
):
    pid_n = tl.program_id(0) # KV block index
    pid_s = tl.program_id(1)
    pid_h = tl.program_id(2)

    stride_tok = tl.num_programs(2) * HEAD_DIM 
    stride_h = tl.num_programs(1) * seq_len
    base_offset = pid_s * stride_tok * seq_len + pid_h * HEAD_DIM

    # Anchor K, V, dK, dV
    k_block = tl.make_block_ptr(base=k_ptr + base_offset, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(pid_n * BLOCK_N, 0), block_shape=(BLOCK_N, HEAD_DIM), order=(1, 0))
    v_block = tl.make_block_ptr(base=v_ptr + base_offset, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(pid_n * BLOCK_N, 0), block_shape=(BLOCK_N, HEAD_DIM), order=(1, 0))
    dk_block = tl.make_block_ptr(base=dk_ptr + base_offset, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(pid_n * BLOCK_N, 0), block_shape=(BLOCK_N, HEAD_DIM), order=(1, 0))
    dv_block = tl.make_block_ptr(base=dv_ptr + base_offset, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(pid_n * BLOCK_N, 0), block_shape=(BLOCK_N, HEAD_DIM), order=(1, 0))

    # Sliding Q and dO
    q_block = tl.make_block_ptr(base=q_ptr + base_offset, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(0, 0), block_shape=(BLOCK_M, HEAD_DIM), order=(1, 0))
    do_block = tl.make_block_ptr(base=do_ptr + base_offset, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(0, 0), block_shape=(BLOCK_M, HEAD_DIM), order=(1, 0))
    
    lse_block = tl.make_block_ptr(base=lse_ptr + pid_h * stride_h + pid_s * seq_len, shape=(seq_len,), strides=(1,), offsets=(0,), block_shape=(BLOCK_M,), order=(0,))
    odo_block = tl.make_block_ptr(base=odo_sum_ptr + pid_h * stride_h + pid_s * seq_len, shape=(seq_len,), strides=(1,), offsets=(0,), block_shape=(BLOCK_M,), order=(0,))

    k = tl.load(k_block, boundary_check=(0,), padding_option="zero")
    v = tl.load(v_block, boundary_check=(0,), padding_option="zero")

    acc_dk = tl.zeros((BLOCK_N, HEAD_DIM), tl.float32)
    acc_dv = tl.zeros((BLOCK_N, HEAD_DIM), tl.float32)
    qk_scale = sm_scale * 1.44269504

    for m in range(0, seq_len, BLOCK_M):
        q = tl.load(q_block, boundary_check=(0,), padding_option="zero")
        do = tl.load(do_block, boundary_check=(0,), padding_option="zero")
        lse = tl.load(lse_block, boundary_check=(0,), padding_option="zero")
        odo = tl.load(odo_block, boundary_check=(0,), padding_option="zero")

        s = tl.dot(q, tl.trans(k)) * qk_scale
        p = tl.math.exp2(s - lse[:, None])
        
        dp = tl.dot(do, tl.trans(v), out_dtype=tl.float32)
        ds = p * (dp - odo[:, None])

        acc_dv += tl.dot(tl.trans(p).to(tl.bfloat16), do, out_dtype=tl.float32)
        acc_dk += tl.dot(tl.trans(ds).to(tl.bfloat16), q, out_dtype=tl.float32)

        q_block = tl.advance(q_block, (BLOCK_M, 0))
        do_block = tl.advance(do_block, (BLOCK_M, 0))
        lse_block = tl.advance(lse_block, (BLOCK_M,))
        odo_block = tl.advance(odo_block, (BLOCK_M,))

    acc_dk = acc_dk * sm_scale
    tl.store(dk_block, acc_dk.to(tl.bfloat16), boundary_check=(0,))
    tl.store(dv_block, acc_dv.to(tl.bfloat16), boundary_check=(0,))

# -----------------------------------------------------------
# Precompute Delta & Wrapper
# -----------------------------------------------------------
@triton.jit
def calc_odo_sum(o_ptr, do_ptr, odo_sum_ptr, seq_len, HEAD_DIM: tl.constexpr, BLOCK_K: tl.constexpr):
    pid = tl.program_id(axis=0)
    pid_s = tl.program_id(axis=1)
    pid_h = tl.program_id(axis=2)

    stride_tok = tl.num_programs(axis=2) * HEAD_DIM
    stride_h = tl.num_programs(axis=1) * seq_len
    o_ptr_b = o_ptr + pid_s * stride_tok * seq_len + pid_h * HEAD_DIM
    do_ptr_b = do_ptr + pid_s * stride_tok * seq_len + pid_h * HEAD_DIM
    
    offset = pid_h * stride_h + pid_s * seq_len + pid * BLOCK_K + tl.arange(0, BLOCK_K)
    mask = offset < (pid_h * stride_h + pid_s * seq_len + seq_len)

    o_block = tl.make_block_ptr(base=o_ptr_b, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(pid * BLOCK_K, 0), block_shape=(BLOCK_K, HEAD_DIM), order=(1, 0))
    do_block = tl.make_block_ptr(base=do_ptr_b, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(pid * BLOCK_K, 0), block_shape=(BLOCK_K, HEAD_DIM), order=(1, 0))

    o = tl.load(o_block, boundary_check=(0,), padding_option="zero")
    do = tl.load(do_block, boundary_check=(0,), padding_option="zero")

    odo = tl.sum(o.to(tl.float32) * do.to(tl.float32), axis=1)
    tl.store(odo_sum_ptr + offset, odo, mask=mask)

def flash_att_backward_split(q, k, v, o, do, lse_sum, num_docs):
    cum_seq_len, heads, head_dim = q.shape
    seq_len = cum_seq_len // num_docs

    dq = torch.empty_like(q)
    dk = torch.empty_like(k)
    dv = torch.empty_like(v)
    odo_sum = torch.empty_like(lse_sum)
    sm_scale = 1 / (head_dim ** 0.5)

    # 1. Delta pre-compute
    calc_odo_sum[(triton.cdiv(seq_len, 256), num_docs, heads)](o, do, odo_sum, seq_len, head_dim, BLOCK_K=256)
    
    # 2. Split Kernels (Launched asynchronously)
    grid_dq = lambda META: (triton.cdiv(seq_len, META['BLOCK_M']), num_docs, heads)
    grid_dkdv = lambda META: (triton.cdiv(seq_len, META['BLOCK_N']), num_docs, heads)

    bwd_kernel_dq[grid_dq](q, k, v, do, dq, lse_sum, odo_sum, seq_len, sm_scale, head_dim)
    bwd_kernel_dk_dv[grid_dkdv](q, k, v, do, dk, dv, lse_sum, odo_sum, seq_len, sm_scale, head_dim)

    return dq, dk, dv

# -----------------------------------------------------------
# Benchmark
# -----------------------------------------------------------
SEQ_LEN = 1024
NUM_DOCS = 8
NUM_HEADS = 12
HEAD_DIM = 128

q = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
k = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
v = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
o = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
do = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
lse_sum = torch.randn(NUM_HEADS, NUM_DOCS*SEQ_LEN, device=DEVICE, dtype=DTYPE)

# Baseline prep
q_ref = q.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
k_ref = k.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
v_ref = v.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
do_ref = do.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone()

with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
    out_ref = F.scaled_dot_product_attention(q_ref, k_ref, v_ref)

def bench_sdpa():
    q_ref.grad = None
    k_ref.grad = None
    v_ref.grad = None
    out_ref.backward(do_ref, retain_graph=True)

# Warmup JIT
_ = flash_att_backward_split(q, k, v, o, do, lse_sum, NUM_DOCS)

ms_flash = triton.testing.do_bench(bench_sdpa)
ms_triton = triton.testing.do_bench(lambda: flash_att_backward_split(q, k, v, o, do, lse_sum, NUM_DOCS))

flops_bwd = 10 * NUM_DOCS * NUM_HEADS * (SEQ_LEN ** 2) * HEAD_DIM
tflops_flash = flops_bwd / (ms_flash * 1e-3) / 1e12
tflops_triton = flops_bwd / (ms_triton * 1e-3) / 1e12

print("\n--- Split-Kernel FA2 Backward Benchmark ---")
print(f"{'Implementation':<15} | {'Time (ms)':>10} | {'TFLOPs':>10}")
print("-" * 42)
print(f"{'FlashAttention2':<15} | {ms_flash:>10.4f} | {tflops_flash:>10.1f}")
print(f"{'Triton FA2 Split':<15} | {ms_triton:>10.4f} | {tflops_triton:>10.1f}")


--- Split-Kernel FA2 Backward Benchmark ---
Implementation  |  Time (ms) |     TFLOPs
------------------------------------------
FlashAttention2 |     1.5763 |       81.7
Triton FA2 Split |     2.0623 |       62.5


In [ ]:

# #Assuming q,k,v to be of shape [num_tokens,num_heads,head_dim]
# @triton.autotune(
#     configs=[
#         # A6000 strict SRAM limit configs (< 99KB)
#         # 1. Balanced blocks, low pipeline
#         triton.Config({'BLOCK_M': 32, 'BLOCK_N': 64}, num_warps=4, num_stages=2),
#         triton.Config({'BLOCK_M': 64, 'BLOCK_N': 32}, num_warps=4, num_stages=2),
        
#         # 2. Maximum square block, NO pipelining (1 stage only)
#         triton.Config({'BLOCK_M': 64, 'BLOCK_N': 64}, num_warps=8, num_stages=1), 
        
#         # 3. Small square, deeper pipeline for latency hiding
#         triton.Config({'BLOCK_M': 32, 'BLOCK_N': 32}, num_warps=4, num_stages=3),
#         triton.Config({'BLOCK_M': 32, 'BLOCK_N': 32}, num_warps=8, num_stages=3),
        
#         # 4. Extreme skew for edge cases
#         triton.Config({'BLOCK_M': 16, 'BLOCK_N': 64}, num_warps=4, num_stages=2),
#         triton.Config({'BLOCK_M': 64, 'BLOCK_N': 16}, num_warps=4, num_stages=2),
#     ],
#     key=['seq_len', 'HEAD_DIM'],
# )
# @triton.jit
# def backwd_dkv(q_ptr,k_ptr,v_ptr,dk_ptr,dv_ptr,do_ptr,lse_ptr,odo_sum_ptr,seq_len,sm_scale:tl.constexpr,
#                        HEAD_DIM:tl.constexpr,BLOCK_M:tl.constexpr,BLOCK_N:tl.constexpr):
    
#     pid=tl.program_id(axis=0) #which tokens in the doc-head
#     pid_s=tl.program_id(axis=1) # which document
#     pid_h=tl.program_id(axis=2) # which head in doc

#     #stride between documents is num_heads*head_dim
#     stride_tok=tl.num_programs(axis=2)*HEAD_DIM 
#     stride_h=tl.num_programs(axis=1)*seq_len
    
#     base_offset=pid_s*stride_tok*seq_len + pid_h*HEAD_DIM

#     # we will store this in form of (heads,docs) for mem coaelscing so different strides
#     lse_ptr_b=lse_ptr+pid_h*stride_h + pid_s*seq_len
#     odo_sum_ptr_b=odo_sum_ptr+pid_h*stride_h + pid_s*seq_len

#     qk_scale = sm_scale * 1.44269504

#     #Load  block pointers for all required tensors in computation
#     q_block_ptr=tl.make_block_ptr(q_ptr + base_offset,shape=(seq_len,HEAD_DIM),strides=(stride_tok,1),
#                                     offsets=(0,00),block_shape=(BLOCK_M,HEAD_DIM),order=(1,0)) 
     
#     k_block_ptr=tl.make_block_ptr(k_ptr + base_offset,shape=(seq_len,HEAD_DIM),strides=(stride_tok,1),
#                                        offsets=(pid*BLOCK_N,0),block_shape=(BLOCK_N,HEAD_DIM),order=(1,0))  

#     v_block_ptr=tl.make_block_ptr(v_ptr + base_offset,shape=(seq_len,HEAD_DIM),strides=(stride_tok,1),
#                                        offsets=(pid*BLOCK_N,0),block_shape=(BLOCK_N,HEAD_DIM),order=(1,0)) 
    
#     do_block_ptr=tl.make_block_ptr(do_ptr + base_offset,shape=(seq_len,HEAD_DIM),strides=(stride_tok,1),
#                                 offsets=(0,0),block_shape=(BLOCK_M,HEAD_DIM),order=(1,0)) 
    
#     dv_block_ptr=tl.make_block_ptr(dv_ptr + base_offset,shape=(seq_len,HEAD_DIM),strides=(stride_tok,1),
#                             offsets=(pid*BLOCK_N,0),block_shape=(BLOCK_N,HEAD_DIM),order=(1,0)) 
    
#     dk_block_ptr=tl.make_block_ptr(dk_ptr + base_offset,shape=(seq_len,HEAD_DIM),strides=(stride_tok,1),
#                             offsets=(pid*BLOCK_N,0),block_shape=(BLOCK_N,HEAD_DIM),order=(1,0)) 

    
#     #strides 0 odo sum and lse ptrs
#     odo_sum_block_ptr=tl.make_block_ptr(base=odo_sum_ptr_b,shape=(seq_len,),
#                                strides=(1,),offsets=(0,),
#                                block_shape=(BLOCK_M,),order=(0,))
    
#     lse_block_ptr=tl.make_block_ptr(base=lse_ptr_b,shape=(seq_len,),
#                             strides=(1,),offsets=(0,),
#                             block_shape=(BLOCK_M,),order=(0,))
    
#     k=tl.load(k_block_ptr,boundary_check=(0,),padding_option="zero")
#     v=tl.load(v_block_ptr,boundary_check=(0,),padding_option="zero")

#     acc_dv=tl.zeros((BLOCK_N,HEAD_DIM),tl.float32)
#     acc_dk=tl.zeros((BLOCK_N,HEAD_DIM),tl.float32)
    

#     for i in range(0,seq_len,BLOCK_M):
#         #laod do dq and q for calculating
#         do=tl.load(do_block_ptr,boundary_check=(0,),padding_option="zero")
#         q=tl.load(q_block_ptr,boundary_check=(0,),padding_option="zero")

#         odo_sum=tl.load(odo_sum_block_ptr,boundary_check=(0,),padding_option="zero")
#         lse=tl.load(lse_block_ptr,boundary_check=(0,),padding_option="zero")

#         #calculate similarlity matrix sij=<qi,kj> shape is of B_MxB_N
#         s=tl.dot(q,tl.trans(k))
#         s=s*qk_scale
        
#         #calculate prob via softmax here lse is log(exp(max)+Z),Z is sum of logits stored in forward
        
#         p=tl.math.exp2(s-lse[:,None])
        
#         # To calcualate dL/daij we have <doi,dvj> 
        
#         dp=tl.dot(do,tl.trans(v))

#         # dL/dsij where sij=<qi,kj> . Using chain rule and total derivative we have dL/dsij =sum k dL/daik.daik/dsij
#         # This is sumk dpij.pij(detlta(j,k)-pik)=pij(dpij-D) where D= <1,dO.O>
        
#         ds=p*(dp.to(tl.bfloat16)-odo_sum[:,None]) #
        
#         #observe that oi=aijvj +k .This gives us dvj= sum_i aijdoi. A^Tdo

#         acc_dv=tl.dot(tl.trans(p).to(tl.bfloat16),do,acc_dv)
#         acc_dk=tl.dot(tl.trans(ds).to(tl.bfloat16),q,acc_dk)
       
#        #advance ptrs
       
#         q_block_ptr=tl.advance(q_block_ptr,(BLOCK_M,0))
#         do_block_ptr=tl.advance(do_block_ptr,(BLOCK_M,0))
#         odo_sum_block_ptr=tl.advance(odo_sum_block_ptr,(BLOCK_M,))
#         lse_block_ptr=tl.advance(lse_block_ptr,(BLOCK_M,))

#     acc_dk=acc_dk*sm_scale
    
#     #store results
    
#     tl.store(dk_block_ptr,acc_dk.to(tl.bfloat16), boundary_check=(0,))
#     tl.store(dv_block_ptr,acc_dv.to(tl.bfloat16), boundary_check=(0,))

# @triton.jit
# def calc_odo_sum(o_ptr,do_ptr,odo_sum_ptr,seq_len,HEAD_DIM:tl.constexpr,BLOCK_K:tl.constexpr):
#     pid=tl.program_id(axis=0)
#     pid_h=tl.program_id(axis=2)
#     pid_s=tl.program_id(axis=1)

#     stride_tok=tl.num_programs(axis=2)*HEAD_DIM
#     stride_h=tl.num_programs(axis=1)*seq_len

#     o_ptr_b=o_ptr+pid_s*stride_tok*seq_len+pid_h*HEAD_DIM
#     do_ptr_b=do_ptr+pid_s*stride_tok*seq_len+pid_h*HEAD_DIM

#     odo_sum_ptr_b=odo_sum_ptr+pid_h*stride_h + pid_s*seq_len + pid*BLOCK_K + tl.arange(0,BLOCK_K)
    
#     offset=pid_h*stride_h + pid_s*seq_len + pid*BLOCK_K + tl.arange(0,BLOCK_K)
#     boundary=pid_h*stride_h + pid_s*seq_len + seq_len
#     mask=offset<boundary

#     o_block_ptr=tl.make_block_ptr(base=o_ptr_b,shape=(seq_len,HEAD_DIM),
#                                strides=(stride_tok,1),offsets=(pid*BLOCK_K,0),
#                                block_shape=(BLOCK_K,HEAD_DIM),order=(1,0))

#     do_block_ptr=tl.make_block_ptr(base=do_ptr_b,shape=(seq_len,HEAD_DIM),
#                                strides=(stride_tok,1),offsets=(pid*BLOCK_K,0),
#                                block_shape=(BLOCK_K,HEAD_DIM),order=(1,0))

    
#     o=tl.load(o_block_ptr,boundary_check=(0,),padding_option="zero")
#     do=tl.load(do_block_ptr,boundary_check=(0,),padding_option="zero")

#     odo=tl.sum(o.to(tl.float32) * do.to(tl.float32), axis=1)


#     tl.store(odo_sum_ptr_b,odo,mask=mask)



# @triton.autotune(
#     configs=[
#         # A6000 strict SRAM limit configs (< 99KB)
#         # 1. Balanced blocks, low pipeline
#         triton.Config({'BLOCK_M': 32, 'BLOCK_N': 64}, num_warps=4, num_stages=2),
#         triton.Config({'BLOCK_M': 64, 'BLOCK_N': 32}, num_warps=4, num_stages=2),
        
#         # 2. Maximum square block, NO pipelining (1 stage only)
#         triton.Config({'BLOCK_M': 64, 'BLOCK_N': 64}, num_warps=8, num_stages=1), 
        
#         # 3. Small square, deeper pipeline for latency hiding
#         triton.Config({'BLOCK_M': 32, 'BLOCK_N': 32}, num_warps=4, num_stages=3),
#         triton.Config({'BLOCK_M': 32, 'BLOCK_N': 32}, num_warps=8, num_stages=3),
        
#         # 4. Extreme skew for edge cases
#         triton.Config({'BLOCK_M': 16, 'BLOCK_N': 64}, num_warps=4, num_stages=2),
#         triton.Config({'BLOCK_M': 64, 'BLOCK_N': 16}, num_warps=4, num_stages=2),
#     ],
#     key=['seq_len', 'HEAD_DIM'],
# )
# @triton.jit
# def backwd_dq(q_ptr,k_ptr,v_ptr,dq_ptr,do_ptr,lse_ptr,odo_sum_ptr,seq_len,sm_scale:tl.constexpr,
#                        HEAD_DIM:tl.constexpr,BLOCK_M:tl.constexpr,BLOCK_N:tl.constexpr):
    
#     pid=tl.program_id(axis=0) #which tokens in the doc-head
#     pid_s=tl.program_id(axis=1) # which document
#     pid_h=tl.program_id(axis=2) # which head in doc

#     #stride between documents is num_heads*head_dim
#     stride_tok=tl.num_programs(axis=2)*HEAD_DIM 
#     stride_h=tl.num_programs(axis=1)*seq_len
    
#     base_offset=pid_s*stride_tok*seq_len + pid_h*HEAD_DIM  

#     # we will store this in form of (heads,docs) for mem coaelscing so different strides
#     lse_ptr_b=lse_ptr+pid_h*stride_h + pid_s*seq_len
#     odo_sum_ptr_b=odo_sum_ptr+pid_h*stride_h + pid_s*seq_len

#     qk_scale = sm_scale * 1.44269504

#     #Load  block pointers for all required tensors in computation
#     q_block_ptr=tl.make_block_ptr(q_ptr + base_offset,shape=(seq_len,HEAD_DIM),strides=(stride_tok,1),
#                                     offsets=(pid*BLOCK_M,0),block_shape=(BLOCK_M,HEAD_DIM),order=(1,0)) 
     
#     k_block_ptr=tl.make_block_ptr(k_ptr + base_offset,shape=(seq_len,HEAD_DIM),strides=(stride_tok,1),
#                                        offsets=(0,0),block_shape=(BLOCK_N,HEAD_DIM),order=(1,0)) 

#     v_block_ptr=tl.make_block_ptr(v_ptr + base_offset,shape=(seq_len,HEAD_DIM),strides=(stride_tok,1),
#                                        offsets=(0,0),block_shape=(BLOCK_N,HEAD_DIM),order=(1,0)) 
    
#     do_block_ptr=tl.make_block_ptr(do_ptr + base_offset,shape=(seq_len,HEAD_DIM),strides=(stride_tok,1),
#                                 offsets=(pid*BLOCK_M,0),block_shape=(BLOCK_M,HEAD_DIM),order=(1,0)) 
    
#     dq_block_ptr=tl.make_block_ptr(dq_ptr + base_offset,shape=(seq_len,HEAD_DIM),strides=(stride_tok,1),
#                         offsets=(pid*BLOCK_M,0),block_shape=(BLOCK_M,HEAD_DIM),order=(1,0)) 
    
#     #strides 0 odo sum and lse ptrs
#     odo_sum_block_ptr=tl.make_block_ptr(base=odo_sum_ptr_b,shape=(seq_len,),
#                                strides=(1,),offsets=(pid*BLOCK_M,),
#                                block_shape=(BLOCK_M,),order=(0,))
    
#     lse_block_ptr=tl.make_block_ptr(base=lse_ptr_b,shape=(seq_len,),
#                             strides=(1,),offsets=(pid*BLOCK_M,),
#                             block_shape=(BLOCK_M,),order=(0,))
    
#     q=tl.load(q_block_ptr,boundary_check=(0,),padding_option="zero")
#     do=tl.load(do_block_ptr,boundary_check=(0,),padding_option="zero")
    
#     odo_sum=tl.load(odo_sum_block_ptr,boundary_check=(0,),padding_option="zero")
#     lse=tl.load(lse_block_ptr,boundary_check=(0,),padding_option="zero")


#     acc_dq=tl.zeros((BLOCK_M,HEAD_DIM),tl.float32)
    
#     for i in range(0,seq_len,BLOCK_N):
#         k=tl.load(k_block_ptr,boundary_check=(0,),padding_option="zero")
#         v=tl.load(v_block_ptr,boundary_check=(0,),padding_option="zero")

#         s=tl.dot(q,tl.trans(k))
#         s=s*qk_scale
        
#         p=tl.math.exp2(s-lse[:,None])
#         dp=tl.dot(do,tl.trans(v),out_dtype=tl.float32)
#         ds=p*(dp-odo_sum[:,None]) #
        
#         acc_dq=tl.dot(ds.to(tl.bfloat16),k,acc_dq)

#         k_block_ptr=tl.advance(k_block_ptr,(BLOCK_N,0))
#         v_block_ptr=tl.advance(v_block_ptr,(BLOCK_N,0))


#     #store results
#     acc_dq=acc_dq*sm_scale
#     tl.store(dq_block_ptr,acc_dq.to(tl.bfloat16), boundary_check=(0,))


# def flash_att_backward_split(q, k, v, o, do, lse_sum, num_docs):
#     cum_seq_len, heads, head_dim = q.shape
#     seq_len = cum_seq_len // num_docs

#     dq = torch.empty_like(q)
#     dk = torch.empty_like(k)
#     dv = torch.empty_like(v)
#     odo_sum = torch.empty_like(lse_sum)
#     sm_scale = 1 / (head_dim ** 0.5)

#     # 1. Delta pre-compute
#     calc_odo_sum[(triton.cdiv(seq_len, 256), num_docs, heads)](o, do, odo_sum, seq_len, head_dim, BLOCK_K=256)
    
#     # 2. Split Kernels (Launched asynchronously)
#     grid_dq = lambda META: (triton.cdiv(seq_len, META['BLOCK_M']), num_docs, heads)
#     grid_dkdv = lambda META: (triton.cdiv(seq_len, META['BLOCK_N']), num_docs, heads)

#     backwd_dq[grid_dq](q, k, v, dq, do, lse_sum, odo_sum, seq_len, sm_scale, head_dim)
#     backwd_dkv[grid_dkdv](q, k, v, dk, dv, do, lse_sum, odo_sum, seq_len, sm_scale, head_dim)

#     return dq, dk, dv


# SEQ_LEN=1024
# NUM_DOCS=8
# NUM_HEADS=12
# HEAD_DIM=128
# DTYPE=torch.bfloat16

# q = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
# k = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
# v = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
# o = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
# do = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
# lse_sum=torch.randn(NUM_HEADS,NUM_DOCS*SEQ_LEN,device=DEVICE, dtype=DTYPE)

# flash_att_backward_split(q,k,v,o,do,lse_sum,NUM_DOCS)
# ms_triton = triton.testing.do_bench(lambda: flash_att_backward_split(q,k,v,o,do,lse_sum,NUM_DOCS)
# )
# ms_triton

# SEQ_LEN = 1024
# NUM_DOCS = 8
# NUM_HEADS = 12
# HEAD_DIM = 128

# q = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
# k = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
# v = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
# o = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
# do = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
# lse_sum = torch.randn(NUM_HEADS, NUM_DOCS*SEQ_LEN, device=DEVICE, dtype=DTYPE)

# # Baseline prep
# q_ref = q.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
# k_ref = k.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
# v_ref = v.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
# do_ref = do.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone()

# with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
#     out_ref = F.scaled_dot_product_attention(q_ref, k_ref, v_ref)

# def bench_sdpa():
#     q_ref.grad = None
#     k_ref.grad = None
#     v_ref.grad = None
#     out_ref.backward(do_ref, retain_graph=True)

# # Warmup JIT
# _ = flash_att_backward_split(q, k, v, o, do, lse_sum, NUM_DOCS)

# ms_flash = triton.testing.do_bench(bench_sdpa)
# ms_triton = triton.testing.do_bench(lambda: flash_att_backward_split(q, k, v, o, do, lse_sum, NUM_DOCS))

# flops_bwd = 10 * NUM_DOCS * NUM_HEADS * (SEQ_LEN ** 2) * HEAD_DIM
# tflops_flash = flops_bwd / (ms_flash * 1e-3) / 1e12
# tflops_triton = flops_bwd / (ms_triton * 1e-3) / 1e12

# print("\n--- Split-Kernel FA2 Backward Benchmark ---")
# print(f"{'Implementation':<15} | {'Time (ms)':>10} | {'TFLOPs':>10}")
# print("-" * 42)
# print(f"{'FlashAttention2':<15} | {ms_flash:>10.4f} | {tflops_flash:>10.1f}")
# print(f"{'Triton FA2 Split':<15} | {ms_triton:>10.4f} | {tflops_triton:>10.1f}")



--- Split-Kernel FA2 Backward Benchmark ---
Implementation  |  Time (ms) |     TFLOPs
------------------------------------------
FlashAttention2 |     1.5742 |       81.8
Triton FA2 Split |     2.1476 |       60.0


In [ ]:
import torch
import triton
import triton.language as tl
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

# Try to import flex_attention (requires PyTorch >= 2.5)
try:
    
    from torch.nn.attention.flex_attention import flex_attention, create_block_mask
    HAS_FLEX = True
except ImportError:
    HAS_FLEX = False
    print("FlexAttention not found. Please upgrade to PyTorch 2.5+")

DEVICE = "cuda:2"
DTYPE = torch.bfloat16

# =====================================================================
# YOUR TRITON KERNELS (Silently patched syntax typos to compile)
# =====================================================================
@triton.autotune(
    configs=[
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 64}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 32}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 64}, num_warps=8, num_stages=1), 
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 32}, num_warps=4, num_stages=3),
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 32}, num_warps=8, num_stages=3),
        triton.Config({'BLOCK_M': 16, 'BLOCK_N': 64}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 16}, num_warps=4, num_stages=2),
    ],
    key=['total_tokens', 'HEAD_DIM'],
)
@triton.jit
def backwd_dkv(q_ptr,k_ptr,v_ptr,dk_ptr,dv_ptr,do_ptr,lse_ptr,odo_sum_ptr,cuseq_ptr,total_tokens,sm_scale:tl.constexpr,
                       HEAD_DIM:tl.constexpr,BLOCK_M:tl.constexpr,BLOCK_N:tl.constexpr):
    pid=tl.program_id(axis=0) 
    pid_s=tl.program_id(axis=1) 
    pid_h=tl.program_id(axis=2) 

    start=tl.load(cuseq_ptr+pid_s)
    end=tl.load(cuseq_ptr+pid_s+1)
    doc_len=end-start

    if pid*BLOCK_N>=doc_len:
        return

    stride_tok=tl.num_programs(axis=2)*HEAD_DIM 
    stride_h=total_tokens
    
    base_offset=start*stride_tok + pid_h*HEAD_DIM

    lse_ptr_b=lse_ptr+pid_h*stride_h + start
    odo_sum_ptr_b=odo_sum_ptr+pid_h*stride_h + start

    qk_scale = sm_scale * 1.44269504

    q_block_ptr=tl.make_block_ptr(q_ptr + base_offset,shape=(doc_len,HEAD_DIM),strides=(stride_tok,1),
                                    offsets=(pid*BLOCK_N,0),block_shape=(BLOCK_M,HEAD_DIM),order=(1,0)) 
    k_block_ptr=tl.make_block_ptr(k_ptr + base_offset,shape=(doc_len,HEAD_DIM),strides=(stride_tok,1),
                                       offsets=(pid*BLOCK_N,0),block_shape=(BLOCK_N,HEAD_DIM),order=(1,0))  
    v_block_ptr=tl.make_block_ptr(v_ptr + base_offset,shape=(doc_len,HEAD_DIM),strides=(stride_tok,1),
                                       offsets=(pid*BLOCK_N,0),block_shape=(BLOCK_N,HEAD_DIM),order=(1,0)) 
    do_block_ptr=tl.make_block_ptr(do_ptr + base_offset,shape=(doc_len,HEAD_DIM),strides=(stride_tok,1),
                                offsets=(pid*BLOCK_N,0),block_shape=(BLOCK_M,HEAD_DIM),order=(1,0)) 
    dv_block_ptr=tl.make_block_ptr(dv_ptr + base_offset,shape=(doc_len,HEAD_DIM),strides=(stride_tok,1),
                            offsets=(pid*BLOCK_N,0),block_shape=(BLOCK_N,HEAD_DIM),order=(1,0)) 
    dk_block_ptr=tl.make_block_ptr(dk_ptr + base_offset,shape=(doc_len,HEAD_DIM),strides=(stride_tok,1),
                            offsets=(pid*BLOCK_N,0),block_shape=(BLOCK_N,HEAD_DIM),order=(1,0)) 

    odo_sum_block_ptr=tl.make_block_ptr(base=odo_sum_ptr_b,shape=(doc_len,),
                               strides=(1,),offsets=(pid*BLOCK_N,),
                               block_shape=(BLOCK_M,),order=(0,))
    lse_block_ptr=tl.make_block_ptr(base=lse_ptr_b,shape=(doc_len,),
                            strides=(1,),offsets=(pid*BLOCK_N,),
                            block_shape=(BLOCK_M,),order=(0,))
    
    k=tl.load(k_block_ptr,boundary_check=(0,),padding_option="zero")
    v=tl.load(v_block_ptr,boundary_check=(0,),padding_option="zero")

    acc_dv=tl.zeros((BLOCK_N,HEAD_DIM),tl.float32)
    acc_dk=tl.zeros((BLOCK_N,HEAD_DIM),tl.float32)
    
    indices_n=pid*BLOCK_N+tl.arange(0,BLOCK_N)

    diagonal_end = tl.minimum((pid + 1) * BLOCK_N, doc_len)

    for i in range(pid*BLOCK_N,diagonal_end,BLOCK_M):
        do=tl.load(do_block_ptr,boundary_check=(0,),padding_option="zero")
        q=tl.load(q_block_ptr,boundary_check=(0,),padding_option="zero")
        odo_sum=tl.load(odo_sum_block_ptr,boundary_check=(0,),padding_option="zero")
        lse=tl.load(lse_block_ptr,boundary_check=(0,),padding_option="zero")

        indices_m=i+tl.arange(0,BLOCK_M)

        s=tl.dot(q,tl.trans(k))
        s=s*qk_scale
        
        att_mask=indices_m[:,None]>=indices_n[None,:]
        doc_mask= indices_n<doc_len
        att_mask=att_mask & doc_mask[None,:]
        s=tl.where(att_mask,s,float("-inf"))

        p=tl.math.exp2(s-lse[:,None])
        dp=tl.dot(do,tl.trans(v))
        ds=p*(dp.to(tl.bfloat16)-odo_sum[:,None])
        
        acc_dv=tl.dot(tl.trans(p).to(tl.bfloat16),do,acc_dv)
        acc_dk=tl.dot(tl.trans(ds).to(tl.bfloat16),q,acc_dk)
       

        q_block_ptr=tl.advance(q_block_ptr,(BLOCK_M,0))
        do_block_ptr=tl.advance(do_block_ptr,(BLOCK_M,0))
        odo_sum_block_ptr=tl.advance(odo_sum_block_ptr,(BLOCK_M,))
        lse_block_ptr=tl.advance(lse_block_ptr,(BLOCK_M,))

    for _ in range(diagonal_end,doc_len,BLOCK_M):
        do=tl.load(do_block_ptr,boundary_check=(0,),padding_option="zero")
        q=tl.load(q_block_ptr,boundary_check=(0,),padding_option="zero")
        odo_sum=tl.load(odo_sum_block_ptr,boundary_check=(0,),padding_option="zero")
        lse=tl.load(lse_block_ptr,boundary_check=(0,),padding_option="zero")

        s=tl.dot(q,tl.trans(k))
        s=s*qk_scale
        p=tl.math.exp2(s-lse[:,None])
        dp=tl.dot(do,tl.trans(v))
        ds=p*(dp.to(tl.bfloat16)-odo_sum[:,None]) 
        
        acc_dv=tl.dot(tl.trans(p).to(tl.bfloat16),do,acc_dv)
        acc_dk=tl.dot(tl.trans(ds).to(tl.bfloat16),q,acc_dk)
       
        q_block_ptr=tl.advance(q_block_ptr,(BLOCK_M,0))
        do_block_ptr=tl.advance(do_block_ptr,(BLOCK_M,0))
        odo_sum_block_ptr=tl.advance(odo_sum_block_ptr,(BLOCK_M,))
        lse_block_ptr=tl.advance(lse_block_ptr,(BLOCK_M,))

    acc_dk=acc_dk*sm_scale
    tl.store(dk_block_ptr,acc_dk.to(tl.bfloat16), boundary_check=(0,))
    tl.store(dv_block_ptr,acc_dv.to(tl.bfloat16), boundary_check=(0,))

@triton.jit
def calc_odo_sum(
    o_ptr, do_ptr, odo_sum_ptr, cuseq_ptr, 
    total_tokens: tl.constexpr, 
    HEAD_DIM: tl.constexpr, 
    BLOCK_K: tl.constexpr
):
    pid = tl.program_id(axis=0)   
    pid_s = tl.program_id(axis=1)  
    pid_h = tl.program_id(axis=2)  

    start = tl.load(cuseq_ptr+pid_s)
    end = tl.load(cuseq_ptr+pid_s+1)
    doc_len = end - start
    
    if pid * BLOCK_K >= doc_len:
        return

    num_heads = tl.num_programs(axis=2)
    stride_tok = num_heads * HEAD_DIM
    stride_h = total_tokens 

    o_ptr_b = o_ptr + start * stride_tok + pid_h * HEAD_DIM
    do_ptr_b = do_ptr + start * stride_tok + pid_h * HEAD_DIM

    token_offsets = pid * BLOCK_K + tl.arange(0, BLOCK_K)
    odo_sum_ptr_b = odo_sum_ptr + pid_h * stride_h + start + token_offsets
    mask = token_offsets < doc_len

    o_block_ptr = tl.make_block_ptr(
        base=o_ptr_b, shape=(doc_len, HEAD_DIM),
        strides=(stride_tok, 1), offsets=(pid * BLOCK_K, 0),
        block_shape=(BLOCK_K, HEAD_DIM), order=(1, 0)
    )

    do_block_ptr = tl.make_block_ptr(
        base=do_ptr_b, shape=(doc_len, HEAD_DIM),
        strides=(stride_tok, 1), offsets=(pid * BLOCK_K, 0),
        block_shape=(BLOCK_K, HEAD_DIM), order=(1, 0)
    )

    o = tl.load(o_block_ptr, boundary_check=(0,), padding_option="zero")
    do = tl.load(do_block_ptr, boundary_check=(0,), padding_option="zero")

    odo = tl.sum(o.to(tl.float32) * do.to(tl.float32), axis=1)
    tl.store(odo_sum_ptr_b, odo, mask=mask)

@triton.autotune(
    configs=[
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 64}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 32}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 64}, num_warps=8, num_stages=1), 
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 32}, num_warps=4, num_stages=3),
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 32}, num_warps=8, num_stages=3),
        triton.Config({'BLOCK_M': 16, 'BLOCK_N': 64}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 16}, num_warps=4, num_stages=2),
    ],
    key=['total_tokens', 'HEAD_DIM'],
)
@triton.jit
def backwd_dq(q_ptr,k_ptr,v_ptr,dq_ptr,do_ptr,lse_ptr,odo_sum_ptr,cuseq_ptr,total_tokens,sm_scale:tl.constexpr,
                       HEAD_DIM:tl.constexpr,BLOCK_M:tl.constexpr,BLOCK_N:tl.constexpr):
    pid=tl.program_id(axis=0) 
    pid_s=tl.program_id(axis=1) 
    pid_h=tl.program_id(axis=2) 

    start=tl.load(cuseq_ptr+pid_s)
    end=tl.load(cuseq_ptr+pid_s+1)
    doc_len=end-start

    if pid*BLOCK_M>=doc_len:
        return

    stride_tok=tl.num_programs(axis=2)*HEAD_DIM 
    stride_h=total_tokens
    
    base_offset=start*stride_tok + pid_h*HEAD_DIM  

    lse_ptr_b=lse_ptr+pid_h*stride_h + start
    odo_sum_ptr_b=odo_sum_ptr+pid_h*stride_h + start



    qk_scale = sm_scale * 1.44269504

    q_block_ptr=tl.make_block_ptr(q_ptr + base_offset,shape=(doc_len,HEAD_DIM),strides=(stride_tok,1),
                                    offsets=(pid*BLOCK_M,0),block_shape=(BLOCK_M,HEAD_DIM),order=(1,0)) 
    k_block_ptr=tl.make_block_ptr(k_ptr + base_offset,shape=(doc_len,HEAD_DIM),strides=(stride_tok,1),
                                       offsets=(0,0),block_shape=(BLOCK_N,HEAD_DIM),order=(1,0)) 
    v_block_ptr=tl.make_block_ptr(v_ptr + base_offset,shape=(doc_len,HEAD_DIM),strides=(stride_tok,1),
                                       offsets=(0,0),block_shape=(BLOCK_N,HEAD_DIM),order=(1,0)) 
    do_block_ptr=tl.make_block_ptr(do_ptr + base_offset,shape=(doc_len,HEAD_DIM),strides=(stride_tok,1),
                                offsets=(pid*BLOCK_M,0),block_shape=(BLOCK_M,HEAD_DIM),order=(1,0)) 
    dq_block_ptr=tl.make_block_ptr(dq_ptr + base_offset,shape=(doc_len,HEAD_DIM),strides=(stride_tok,1),
                        offsets=(pid*BLOCK_M,0),block_shape=(BLOCK_M,HEAD_DIM),order=(1,0)) 
    
    odo_sum_block_ptr=tl.make_block_ptr(base=odo_sum_ptr_b,shape=(doc_len,),
                               strides=(1,),offsets=(pid*BLOCK_M,),
                               block_shape=(BLOCK_M,),order=(0,))
    lse_block_ptr=tl.make_block_ptr(base=lse_ptr_b,shape=(doc_len,),
                            strides=(1,),offsets=(pid*BLOCK_M,),
                            block_shape=(BLOCK_M,),order=(0,))
    
    q=tl.load(q_block_ptr,boundary_check=(0,),padding_option="zero")
    do=tl.load(do_block_ptr,boundary_check=(0,),padding_option="zero")
    odo_sum=tl.load(odo_sum_block_ptr,boundary_check=(0,),padding_option="zero")
    lse=tl.load(lse_block_ptr,boundary_check=(0,),padding_option="zero")

    acc_dq=tl.zeros((BLOCK_M,HEAD_DIM),tl.float32)

    for i in range(0,pid*BLOCK_M,BLOCK_N):
        k=tl.load(k_block_ptr,boundary_check=(0,),padding_option="zero")
        v=tl.load(v_block_ptr,boundary_check=(0,),padding_option="zero")

        s=tl.dot(q,tl.trans(k))
        s=s*qk_scale
        
        p=tl.math.exp2(s-lse[:,None])
        dp=tl.dot(do,tl.trans(v),out_dtype=tl.float32)
        ds=p*(dp-odo_sum[:,None])
        
        acc_dq=tl.dot(ds.to(tl.bfloat16),k,acc_dq)

        k_block_ptr=tl.advance(k_block_ptr,(BLOCK_N,0))
        v_block_ptr=tl.advance(v_block_ptr,(BLOCK_N,0))

    indices_m=pid*BLOCK_M + tl.arange(0,BLOCK_M)

    diagonal_end = tl.minimum((pid + 1) * BLOCK_M, doc_len)

    for i in range(pid*BLOCK_M,diagonal_end,BLOCK_N):
        k=tl.load(k_block_ptr,boundary_check=(0,),padding_option="zero")
        v=tl.load(v_block_ptr,boundary_check=(0,),padding_option="zero")

        s=tl.dot(q,tl.trans(k))
        s=s*qk_scale
        
        indices_n=i+tl.arange(0,BLOCK_N)

        att_mask=indices_m[:,None]>=indices_n[None,:]
        doc_mask= indices_n<doc_len
        att_mask=att_mask & doc_mask[None,:]
        s=tl.where(att_mask,s,float("-inf"))

        p=tl.math.exp2(s-lse[:,None])
        dp=tl.dot(do,tl.trans(v),out_dtype=tl.float32)
        ds=p*(dp-odo_sum[:,None])
        
        acc_dq=tl.dot(ds.to(tl.bfloat16),k,acc_dq)

        k_block_ptr=tl.advance(k_block_ptr,(BLOCK_N,0))
        v_block_ptr=tl.advance(v_block_ptr,(BLOCK_N,0))

    acc_dq=acc_dq*sm_scale
    tl.store(dq_block_ptr,acc_dq.to(tl.bfloat16), boundary_check=(0,))

def flash_att_backward_split(q, k, v, o, do, lse_sum, num_docs, cuseq):
    total_tokens, heads, head_dim = q.shape
    max_seq_len = 1024

    dq = torch.empty_like(q)
    dk = torch.empty_like(k)
    dv = torch.empty_like(v)
    odo_sum = torch.empty_like(lse_sum)
    lse_sum=lse_sum* 1.44269504
    sm_scale = 1 / (head_dim ** 0.5)

    calc_odo_sum[(triton.cdiv(max_seq_len, 256), num_docs, heads)](o, do, odo_sum, cuseq, total_tokens, head_dim, BLOCK_K=256)
    
    grid_dq = lambda META: (triton.cdiv(max_seq_len, META['BLOCK_M']), num_docs, heads)
    grid_dkdv = lambda META: (triton.cdiv(max_seq_len, META['BLOCK_N']), num_docs, heads)

    backwd_dq[grid_dq](q, k, v, dq, do, lse_sum, odo_sum, cuseq, total_tokens, sm_scale, head_dim)
    backwd_dkv[grid_dkdv](q, k, v, dk, dv, do, lse_sum, odo_sum, cuseq, total_tokens, sm_scale, head_dim)

    return dq, dk, dv

# =====================================================================
# BENCHMARK AND VALIDATION SETUP
# =====================================================================
SEQ_LEN = 1024
NUM_DOCS = 8
NUM_HEADS = 12
HEAD_DIM = 128
TOTAL_TOKENS = NUM_DOCS * SEQ_LEN
DEVICE="cuda:0"
torch.set_default_device(DEVICE)

print("Generating Exact Mathematical Tensors...")
# 1. Base input gradients
q = torch.randn(TOTAL_TOKENS, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
k = torch.randn(TOTAL_TOKENS, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
v = torch.randn(TOTAL_TOKENS, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
do = torch.randn(TOTAL_TOKENS, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
cuseq = torch.arange(0, (NUM_DOCS + 1) * SEQ_LEN, SEQ_LEN, dtype=torch.int32, device=DEVICE)

# 2. Math Generator to get valid `o` and `lse_sum`
q_val = q.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).float()
k_val = k.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).float()
v_val = v.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).float()

sm_scale = 1.0 / (HEAD_DIM ** 0.5)
scores = torch.matmul(q_val, k_val.transpose(-2, -1)) * sm_scale
causal_mask = torch.tril(torch.ones(SEQ_LEN, SEQ_LEN, device=DEVICE)) == 0
scores.masked_fill_(causal_mask, float('-inf'))

m = scores.max(dim=-1, keepdim=True)[0]
scores_shifted = scores - m
exp_scores = torch.exp(scores_shifted)
sum_exp = exp_scores.sum(dim=-1, keepdim=True)
probs = exp_scores / sum_exp

o_exact = torch.matmul(probs, v_val).to(DTYPE)
lse_exact = (m + torch.log(sum_exp)).squeeze(-1).to(DTYPE)

o = o_exact.transpose(1, 2).reshape(TOTAL_TOKENS, NUM_HEADS, HEAD_DIM)
lse_sum = lse_exact.transpose(0, 1).reshape(NUM_HEADS, TOTAL_TOKENS)

# ---------------------------------------------------------
# PyTorch SDPA Reference
# ---------------------------------------------------------
print("Warming up SDPA...")
q_sdpa = q.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
k_sdpa = k.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
v_sdpa = v.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
do_sdpa = do.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone()

with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
    out_sdpa = F.scaled_dot_product_attention(q_sdpa, k_sdpa, v_sdpa, is_causal=True)

out_sdpa.backward(do_sdpa)
dq_sdpa, dk_sdpa, dv_sdpa = q_sdpa.grad, k_sdpa.grad, v_sdpa.grad

def bench_sdpa():
    q_sdpa.grad, k_sdpa.grad, v_sdpa.grad = None, None, None
    out_sdpa.backward(do_sdpa, retain_graph=True)

# ---------------------------------------------------------
# PyTorch FlexAttention
# ---------------------------------------------------------
if HAS_FLEX:
    print("Compiling FlexAttention...")
    def causal_mask_fn(b, h, q_idx, kv_idx):
        return q_idx >= kv_idx

    block_mask = create_block_mask(causal_mask_fn, B=None, H=None, Q_LEN=SEQ_LEN, KV_LEN=SEQ_LEN, device=DEVICE)

    q_flex = q.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
    k_flex = k.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
    v_flex = v.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
    
    compiled_flex = torch.compile(flex_attention)
    out_flex = compiled_flex(q_flex, k_flex, v_flex, block_mask=block_mask)
    
    out_flex.backward(do_sdpa)
    dq_flex, dk_flex, dv_flex = q_flex.grad, k_flex.grad, v_flex.grad

    def bench_flex():
        q_flex.grad, k_flex.grad, v_flex.grad = None, None, None
        out_flex.backward(do_sdpa, retain_graph=True)

# ---------------------------------------------------------
# Your Triton Split Kernels
# ---------------------------------------------------------
print("Warming up Custom Triton...")
dq_tri, dk_tri, dv_tri = flash_att_backward_split(q, k, v, o, do, lse_sum, NUM_DOCS, cuseq)

dq_tri_fmt = dq_tri.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2)
dk_tri_fmt = dk_tri.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2)
dv_tri_fmt = dv_tri.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2)


# =====================================================================
# VERIFICATION & TIMING RESULTS
# =====================================================================
print("\n--- Correctness Check (Max Absolute Diff against SDPA) ---")
if HAS_FLEX:
    print(f"Flex vs SDPA -> dQ: {torch.max(torch.abs(dq_sdpa - dq_flex)):.5f}, dK: {torch.max(torch.abs(dk_sdpa - dk_flex)):.5f}, dV: {torch.max(torch.abs(dv_sdpa - dv_flex)):.5f}")
print(f"Tri vs SDPA  -> dQ: {torch.max(torch.abs(dq_sdpa - dq_tri_fmt)):.5f}, dK: {torch.max(torch.abs(dk_sdpa - dk_tri_fmt)):.5f}, dV: {torch.max(torch.abs(dv_sdpa - dv_tri_fmt)):.5f}")
print("*(Values below ~0.05 are considered numerically equivalent due to bf16 summation differences)*")

print("\n--- Running Benchmarks ---")
ms_flash = triton.testing.do_bench(bench_sdpa)
ms_triton = triton.testing.do_bench(lambda: flash_att_backward_split(q, k, v, o, do, lse_sum, NUM_DOCS, cuseq))
if HAS_FLEX:
    ms_flex = triton.testing.do_bench(bench_flex)

flops_bwd = 10 * NUM_DOCS * NUM_HEADS * (SEQ_LEN ** 2) * HEAD_DIM
tflops_flash = flops_bwd / (ms_flash * 1e-3) / 1e12
tflops_triton = flops_bwd / (ms_triton * 1e-3) / 1e12

print("\n--- Backward Pass Benchmark ---")
print(f"{'Implementation':<17} | {'Time (ms)':>10} | {'TFLOPs':>10}")
print("-" * 45)
print(f"{'SDPA FlashAttn-2':<17} | {ms_flash:>10.4f} | {tflops_flash:>10.1f}")
if HAS_FLEX:
    tflops_flex = flops_bwd / (ms_flex * 1e-3) / 1e12
    print(f"{'FlexAttention':<17} | {ms_flex:>10.4f} | {tflops_flex:>10.1f}")
print(f"{'Triton Custom':<17} | {ms_triton:>10.4f} | {tflops_triton:>10.1f}")

Generating Exact Mathematical Tensors...
Warming up SDPA...
Compiling FlexAttention...
Warming up Custom Triton...


ValueError: Pointer argument (at 0) cannot be accessed from Triton (cpu tensor?)

In [2]:
import torch
import triton
import triton.language as tl
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

# Try to import flex_attention (requires PyTorch >= 2.5)
try:
    from torch.nn.attention.flex_attention import flex_attention, create_block_mask
    HAS_FLEX = True
except ImportError:
    HAS_FLEX = False
    print("FlexAttention not found. Please upgrade to PyTorch 2.5+")

# Set context so Triton doesn't throw the "CPU Tensor" error
DEVICE = "cuda:0"
torch.cuda.set_device(DEVICE)
DTYPE = torch.bfloat16

# =====================================================================
# YOUR TRITON KERNELS (With the 2 Fatal Bugs Patched)
# =====================================================================
@triton.autotune(
    configs=[
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 64}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 32}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 64}, num_warps=8, num_stages=1), 
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 32}, num_warps=4, num_stages=3),
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 32}, num_warps=8, num_stages=3),
        triton.Config({'BLOCK_M': 16, 'BLOCK_N': 64}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 16}, num_warps=4, num_stages=2),
    ],
    key=['total_tokens', 'HEAD_DIM'],
)
@triton.jit
def backwd_dkv(q_ptr,k_ptr,v_ptr,dk_ptr,dv_ptr,do_ptr,lse_ptr,odo_sum_ptr,cuseq_ptr,total_tokens,sm_scale:tl.constexpr,
                       HEAD_DIM:tl.constexpr,BLOCK_M:tl.constexpr,BLOCK_N:tl.constexpr):
    pid=tl.program_id(axis=0) 
    pid_s=tl.program_id(axis=1) 
    pid_h=tl.program_id(axis=2) 

    start=tl.load(cuseq_ptr+pid_s)
    end=tl.load(cuseq_ptr+pid_s+1)
    doc_len=end-start

    # FIX 1: Missing Early Exit Added
    if pid * BLOCK_N >= doc_len:
        return

    stride_tok=tl.num_programs(axis=2)*HEAD_DIM 
    stride_h=total_tokens
    
    base_offset=start*stride_tok + pid_h*HEAD_DIM

    lse_ptr_b=lse_ptr+pid_h*stride_h + start
    odo_sum_ptr_b=odo_sum_ptr+pid_h*stride_h + start

    qk_scale = sm_scale * 1.44269504

    q_block_ptr=tl.make_block_ptr(q_ptr + base_offset,shape=(doc_len,HEAD_DIM),strides=(stride_tok,1),
                                    offsets=(pid*BLOCK_N,0),block_shape=(BLOCK_M,HEAD_DIM),order=(1,0)) 
    k_block_ptr=tl.make_block_ptr(k_ptr + base_offset,shape=(doc_len,HEAD_DIM),strides=(stride_tok,1),
                                       offsets=(pid*BLOCK_N,0),block_shape=(BLOCK_N,HEAD_DIM),order=(1,0))  
    v_block_ptr=tl.make_block_ptr(v_ptr + base_offset,shape=(doc_len,HEAD_DIM),strides=(stride_tok,1),
                                       offsets=(pid*BLOCK_N,0),block_shape=(BLOCK_N,HEAD_DIM),order=(1,0)) 
    do_block_ptr=tl.make_block_ptr(do_ptr + base_offset,shape=(doc_len,HEAD_DIM),strides=(stride_tok,1),
                                offsets=(pid*BLOCK_N,0),block_shape=(BLOCK_M,HEAD_DIM),order=(1,0)) 
    dv_block_ptr=tl.make_block_ptr(dv_ptr + base_offset,shape=(doc_len,HEAD_DIM),strides=(stride_tok,1),
                            offsets=(pid*BLOCK_N,0),block_shape=(BLOCK_N,HEAD_DIM),order=(1,0)) 
    dk_block_ptr=tl.make_block_ptr(dk_ptr + base_offset,shape=(doc_len,HEAD_DIM),strides=(stride_tok,1),
                            offsets=(pid*BLOCK_N,0),block_shape=(BLOCK_N,HEAD_DIM),order=(1,0)) 

    odo_sum_block_ptr=tl.make_block_ptr(base=odo_sum_ptr_b,shape=(doc_len,),
                               strides=(1,),offsets=(pid*BLOCK_N,),
                               block_shape=(BLOCK_M,),order=(0,))
    lse_block_ptr=tl.make_block_ptr(base=lse_ptr_b,shape=(doc_len,),
                            strides=(1,),offsets=(pid*BLOCK_N,),
                            block_shape=(BLOCK_M,),order=(0,))
    
    k=tl.load(k_block_ptr,boundary_check=(0,),padding_option="zero")
    v=tl.load(v_block_ptr,boundary_check=(0,),padding_option="zero")

    acc_dv=tl.zeros((BLOCK_N,HEAD_DIM),tl.float32)
    acc_dk=tl.zeros((BLOCK_N,HEAD_DIM),tl.float32)
    
    indices_n=pid*BLOCK_N+tl.arange(0,BLOCK_N)

    diagonal_end = tl.minimum((pid + 1) * BLOCK_N, doc_len)

    for i in range(pid*BLOCK_N,diagonal_end,BLOCK_M):
        do=tl.load(do_block_ptr,boundary_check=(0,),padding_option="zero")
        q=tl.load(q_block_ptr,boundary_check=(0,),padding_option="zero")
        odo_sum=tl.load(odo_sum_block_ptr,boundary_check=(0,),padding_option="zero")
        lse=tl.load(lse_block_ptr,boundary_check=(0,),padding_option="zero")

        indices_m=i+tl.arange(0,BLOCK_M)

        s=tl.dot(q,tl.trans(k))
        s=s*qk_scale
        
        att_mask=indices_m[:,None]>=indices_n[None,:]
        doc_mask= indices_n<doc_len
        att_mask=att_mask & doc_mask[None,:]
        s=tl.where(att_mask,s,float("-inf"))

        p=tl.math.exp2(s-lse[:,None])
        dp=tl.dot(do,tl.trans(v))
        ds=p*(dp.to(tl.bfloat16)-odo_sum[:,None])
        
        acc_dv=tl.dot(tl.trans(p).to(tl.bfloat16),do,acc_dv)
        acc_dk=tl.dot(tl.trans(ds).to(tl.bfloat16),q,acc_dk)
       
        q_block_ptr=tl.advance(q_block_ptr,(BLOCK_M,0))
        do_block_ptr=tl.advance(do_block_ptr,(BLOCK_M,0))
        odo_sum_block_ptr=tl.advance(odo_sum_block_ptr,(BLOCK_M,))
        lse_block_ptr=tl.advance(lse_block_ptr,(BLOCK_M,))

    for _ in range(diagonal_end,doc_len,BLOCK_M):
        do=tl.load(do_block_ptr,boundary_check=(0,),padding_option="zero")
        q=tl.load(q_block_ptr,boundary_check=(0,),padding_option="zero")
        odo_sum=tl.load(odo_sum_block_ptr,boundary_check=(0,),padding_option="zero")
        lse=tl.load(lse_block_ptr,boundary_check=(0,),padding_option="zero")

        s=tl.dot(q,tl.trans(k))
        s=s*qk_scale
        p=tl.math.exp2(s-lse[:,None])
        dp=tl.dot(do,tl.trans(v))
        ds=p*(dp.to(tl.bfloat16)-odo_sum[:,None]) 
        
        acc_dv=tl.dot(tl.trans(p).to(tl.bfloat16),do,acc_dv)
        acc_dk=tl.dot(tl.trans(ds).to(tl.bfloat16),q,acc_dk)
       
        q_block_ptr=tl.advance(q_block_ptr,(BLOCK_M,0))
        do_block_ptr=tl.advance(do_block_ptr,(BLOCK_M,0))
        odo_sum_block_ptr=tl.advance(odo_sum_block_ptr,(BLOCK_M,))
        lse_block_ptr=tl.advance(lse_block_ptr,(BLOCK_M,))

    acc_dk=acc_dk*sm_scale
    tl.store(dk_block_ptr,acc_dk.to(tl.bfloat16), boundary_check=(0,))
    tl.store(dv_block_ptr,acc_dv.to(tl.bfloat16), boundary_check=(0,))

@triton.jit
def calc_odo_sum(
    o_ptr, do_ptr, odo_sum_ptr, cuseq_ptr, 
    total_tokens: tl.constexpr, 
    HEAD_DIM: tl.constexpr, 
    BLOCK_K: tl.constexpr
):
    pid = tl.program_id(axis=0)   
    pid_s = tl.program_id(axis=1)  
    pid_h = tl.program_id(axis=2)  

    start = tl.load(cuseq_ptr+pid_s)
    end = tl.load(cuseq_ptr+pid_s+1)
    doc_len = end - start
    
    if pid * BLOCK_K >= doc_len:
        return

    num_heads = tl.num_programs(axis=2)
    stride_tok = num_heads * HEAD_DIM
    stride_h = total_tokens 

    o_ptr_b = o_ptr + start * stride_tok + pid_h * HEAD_DIM
    do_ptr_b = do_ptr + start * stride_tok + pid_h * HEAD_DIM

    token_offsets = pid * BLOCK_K + tl.arange(0, BLOCK_K)
    odo_sum_ptr_b = odo_sum_ptr + pid_h * stride_h + start + token_offsets
    mask = token_offsets < doc_len

    o_block_ptr = tl.make_block_ptr(
        base=o_ptr_b, shape=(doc_len, HEAD_DIM),
        strides=(stride_tok, 1), offsets=(pid * BLOCK_K, 0),
        block_shape=(BLOCK_K, HEAD_DIM), order=(1, 0)
    )

    do_block_ptr = tl.make_block_ptr(
        base=do_ptr_b, shape=(doc_len, HEAD_DIM),
        strides=(stride_tok, 1), offsets=(pid * BLOCK_K, 0),
        block_shape=(BLOCK_K, HEAD_DIM), order=(1, 0)
    )

    o = tl.load(o_block_ptr, boundary_check=(0,), padding_option="zero")
    do = tl.load(do_block_ptr, boundary_check=(0,), padding_option="zero")

    odo = tl.sum(o.to(tl.float32) * do.to(tl.float32), axis=1)
    tl.store(odo_sum_ptr_b, odo, mask=mask)

@triton.autotune(
    configs=[
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 64}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 32}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 64}, num_warps=8, num_stages=1), 
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 32}, num_warps=4, num_stages=3),
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 32}, num_warps=8, num_stages=3),
        triton.Config({'BLOCK_M': 16, 'BLOCK_N': 64}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 16}, num_warps=4, num_stages=2),
    ],
    key=['total_tokens', 'HEAD_DIM'],
)
@triton.jit
def backwd_dq(q_ptr,k_ptr,v_ptr,dq_ptr,do_ptr,lse_ptr,odo_sum_ptr,cuseq_ptr,total_tokens,sm_scale:tl.constexpr,
                       HEAD_DIM:tl.constexpr,BLOCK_M:tl.constexpr,BLOCK_N:tl.constexpr):
    pid=tl.program_id(axis=0) 
    pid_s=tl.program_id(axis=1) 
    pid_h=tl.program_id(axis=2) 

    start=tl.load(cuseq_ptr+pid_s)
    end=tl.load(cuseq_ptr+pid_s+1)
    doc_len=end-start

    # FIX 1: Missing Early Exit Added
    if pid * BLOCK_M >= doc_len:
        return

    stride_tok=tl.num_programs(axis=2)*HEAD_DIM 
    stride_h=total_tokens
    
    base_offset=start*stride_tok + pid_h*HEAD_DIM  

    # FIX 2: Corrected pid_s * total_tokens to start
    lse_ptr_b=lse_ptr+pid_h*stride_h + start
    odo_sum_ptr_b=odo_sum_ptr+pid_h*stride_h + start

    qk_scale = sm_scale * 1.44269504

    q_block_ptr=tl.make_block_ptr(q_ptr + base_offset,shape=(doc_len,HEAD_DIM),strides=(stride_tok,1),
                                    offsets=(pid*BLOCK_M,0),block_shape=(BLOCK_M,HEAD_DIM),order=(1,0)) 
    k_block_ptr=tl.make_block_ptr(k_ptr + base_offset,shape=(doc_len,HEAD_DIM),strides=(stride_tok,1),
                                       offsets=(0,0),block_shape=(BLOCK_N,HEAD_DIM),order=(1,0)) 
    v_block_ptr=tl.make_block_ptr(v_ptr + base_offset,shape=(doc_len,HEAD_DIM),strides=(stride_tok,1),
                                       offsets=(0,0),block_shape=(BLOCK_N,HEAD_DIM),order=(1,0)) 
    do_block_ptr=tl.make_block_ptr(do_ptr + base_offset,shape=(doc_len,HEAD_DIM),strides=(stride_tok,1),
                                offsets=(pid*BLOCK_M,0),block_shape=(BLOCK_M,HEAD_DIM),order=(1,0)) 
    dq_block_ptr=tl.make_block_ptr(dq_ptr + base_offset,shape=(doc_len,HEAD_DIM),strides=(stride_tok,1),
                        offsets=(pid*BLOCK_M,0),block_shape=(BLOCK_M,HEAD_DIM),order=(1,0)) 
    
    odo_sum_block_ptr=tl.make_block_ptr(base=odo_sum_ptr_b,shape=(doc_len,),
                               strides=(1,),offsets=(pid*BLOCK_M,),
                               block_shape=(BLOCK_M,),order=(0,))
    lse_block_ptr=tl.make_block_ptr(base=lse_ptr_b,shape=(doc_len,),
                            strides=(1,),offsets=(pid*BLOCK_M,),
                            block_shape=(BLOCK_M,),order=(0,))
    
    q=tl.load(q_block_ptr,boundary_check=(0,),padding_option="zero")
    do=tl.load(do_block_ptr,boundary_check=(0,),padding_option="zero")
    odo_sum=tl.load(odo_sum_block_ptr,boundary_check=(0,),padding_option="zero")
    lse=tl.load(lse_block_ptr,boundary_check=(0,),padding_option="zero")

    acc_dq=tl.zeros((BLOCK_M,HEAD_DIM),tl.float32)

    for i in range(0,pid*BLOCK_M,BLOCK_N):
        k=tl.load(k_block_ptr,boundary_check=(0,),padding_option="zero")
        v=tl.load(v_block_ptr,boundary_check=(0,),padding_option="zero")

        s=tl.dot(q,tl.trans(k))
        s=s*qk_scale
        
        p=tl.math.exp2(s-lse[:,None])
        dp=tl.dot(do,tl.trans(v),out_dtype=tl.float32)
        ds=p*(dp-odo_sum[:,None])
        
        acc_dq=tl.dot(ds.to(tl.bfloat16),k,acc_dq)

        k_block_ptr=tl.advance(k_block_ptr,(BLOCK_N,0))
        v_block_ptr=tl.advance(v_block_ptr,(BLOCK_N,0))

    indices_m=pid*BLOCK_M + tl.arange(0,BLOCK_M)
    diagonal_end = tl.minimum((pid + 1) * BLOCK_M, doc_len)

    for i in range(pid*BLOCK_M,diagonal_end,BLOCK_N):
        k=tl.load(k_block_ptr,boundary_check=(0,),padding_option="zero")
        v=tl.load(v_block_ptr,boundary_check=(0,),padding_option="zero")

        s=tl.dot(q,tl.trans(k))
        s=s*qk_scale
        
        indices_n=i+tl.arange(0,BLOCK_N)

        att_mask=indices_m[:,None]>=indices_n[None,:]
        doc_mask= indices_n<doc_len
        att_mask=att_mask & doc_mask[None,:]
        s=tl.where(att_mask,s,float("-inf"))

        p=tl.math.exp2(s-lse[:,None])
        dp=tl.dot(do,tl.trans(v),out_dtype=tl.float32)
        ds=p*(dp-odo_sum[:,None])
        
        acc_dq=tl.dot(ds.to(tl.bfloat16),k,acc_dq)

        k_block_ptr=tl.advance(k_block_ptr,(BLOCK_N,0))
        v_block_ptr=tl.advance(v_block_ptr,(BLOCK_N,0))

    acc_dq=acc_dq*sm_scale
    tl.store(dq_block_ptr,acc_dq.to(tl.bfloat16), boundary_check=(0,))

def flash_att_backward_split(q, k, v, o, do, lse_sum, num_docs, cuseq):
    total_tokens, heads, head_dim = q.shape
    max_seq_len = 1024

    dq = torch.empty_like(q)
    dk = torch.empty_like(k)
    dv = torch.empty_like(v)
    odo_sum = torch.empty_like(lse_sum)
    sm_scale = 1 / (head_dim ** 0.5)

    lse_sum = lse_sum * 1.44269504

    calc_odo_sum[(triton.cdiv(max_seq_len, 256), num_docs, heads)](o, do, odo_sum, cuseq, total_tokens, head_dim, BLOCK_K=256)
    
    grid_dq = lambda META: (triton.cdiv(max_seq_len, META['BLOCK_M']), num_docs, heads)
    grid_dkdv = lambda META: (triton.cdiv(max_seq_len, META['BLOCK_N']), num_docs, heads)

    backwd_dq[grid_dq](q, k, v, dq, do, lse_sum, odo_sum, cuseq, total_tokens, sm_scale, head_dim)
    backwd_dkv[grid_dkdv](q, k, v, dk, dv, do, lse_sum, odo_sum, cuseq, total_tokens, sm_scale, head_dim)

    return dq, dk, dv

# =====================================================================
# BENCHMARK AND VALIDATION SETUP
# =====================================================================
SEQ_LEN = 1024
NUM_DOCS = 8
NUM_HEADS = 12
HEAD_DIM = 128
TOTAL_TOKENS = NUM_DOCS * SEQ_LEN

print("Generating Exact Mathematical Tensors...")
# 1. Base input gradients
q = torch.randn(TOTAL_TOKENS, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
k = torch.randn(TOTAL_TOKENS, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
v = torch.randn(TOTAL_TOKENS, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
do = torch.randn(TOTAL_TOKENS, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
cuseq = torch.arange(0, (NUM_DOCS + 1) * SEQ_LEN, SEQ_LEN, dtype=torch.int32, device=DEVICE)

# 2. Math Generator to get valid `o` and `lse_sum`
q_val = q.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).float()
k_val = k.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).float()
v_val = v.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).float()

sm_scale = 1.0 / (HEAD_DIM ** 0.5)
scores = torch.matmul(q_val, k_val.transpose(-2, -1)) * sm_scale
causal_mask = torch.tril(torch.ones(SEQ_LEN, SEQ_LEN, device=DEVICE)) == 0
scores.masked_fill_(causal_mask, float('-inf'))

m = scores.max(dim=-1, keepdim=True)[0]
scores_shifted = scores - m
exp_scores = torch.exp(scores_shifted)
sum_exp = exp_scores.sum(dim=-1, keepdim=True)
probs = exp_scores / sum_exp

o_exact = torch.matmul(probs, v_val).to(DTYPE)
lse_exact = (m + torch.log(sum_exp)).squeeze(-1).to(DTYPE)

o = o_exact.transpose(1, 2).reshape(TOTAL_TOKENS, NUM_HEADS, HEAD_DIM)
lse_sum = lse_exact.transpose(0, 1).reshape(NUM_HEADS, TOTAL_TOKENS)

# ---------------------------------------------------------
# PyTorch SDPA Reference
# ---------------------------------------------------------
print("Warming up SDPA...")
q_sdpa = q.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
k_sdpa = k.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
v_sdpa = v.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
do_sdpa = do.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone()

with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
    out_sdpa = F.scaled_dot_product_attention(q_sdpa, k_sdpa, v_sdpa, is_causal=True)

out_sdpa.backward(do_sdpa,retain_graph=True)
dq_sdpa, dk_sdpa, dv_sdpa = q_sdpa.grad, k_sdpa.grad, v_sdpa.grad

def bench_sdpa():
    q_sdpa.grad, k_sdpa.grad, v_sdpa.grad = None, None, None
    out_sdpa.backward(do_sdpa, retain_graph=True)

# ---------------------------------------------------------
# PyTorch FlexAttention
# ---------------------------------------------------------
if HAS_FLEX:
    print("Compiling FlexAttention...")
    def causal_mask_fn(b, h, q_idx, kv_idx):
        return q_idx >= kv_idx

    block_mask = create_block_mask(causal_mask_fn, B=None, H=None, Q_LEN=SEQ_LEN, KV_LEN=SEQ_LEN, device=DEVICE)

    q_flex = q.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
    k_flex = k.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
    v_flex = v.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
    
    compiled_flex = torch.compile(flex_attention)
    out_flex = compiled_flex(q_flex, k_flex, v_flex, block_mask=block_mask)
    
    out_flex.backward(do_sdpa,retain_graph=True)
    dq_flex, dk_flex, dv_flex = q_flex.grad, k_flex.grad, v_flex.grad

    def bench_flex():
        q_flex.grad, k_flex.grad, v_flex.grad = None, None, None
        out_flex.backward(do_sdpa, retain_graph=True)

# ---------------------------------------------------------
# Your Triton Split Kernels
# ---------------------------------------------------------
print("Warming up Custom Triton...")
dq_tri, dk_tri, dv_tri = flash_att_backward_split(q, k, v, o, do, lse_sum, NUM_DOCS, cuseq)

dq_tri_fmt = dq_tri.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2)
dk_tri_fmt = dk_tri.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2)
dv_tri_fmt = dv_tri.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2)


# =====================================================================
# VERIFICATION & TIMING RESULTS
# =====================================================================
print("\n--- Correctness Check (Max Absolute Diff against SDPA) ---")
if HAS_FLEX:
    print(f"Flex vs SDPA -> dQ: {torch.max(torch.abs(dq_sdpa - dq_flex)):.5f}, dK: {torch.max(torch.abs(dk_sdpa - dk_flex)):.5f}, dV: {torch.max(torch.abs(dv_sdpa - dv_flex)):.5f}")
print(f"Tri vs SDPA  -> dQ: {torch.max(torch.abs(dq_sdpa - dq_tri_fmt)):.5f}, dK: {torch.max(torch.abs(dk_sdpa - dk_tri_fmt)):.5f}, dV: {torch.max(torch.abs(dv_sdpa - dv_tri_fmt)):.5f}")
print("*(Values below ~0.05 are considered numerically equivalent due to bf16 summation differences)*")

print("\n--- Running Benchmarks ---")
ms_flash = triton.testing.do_bench(bench_sdpa)
ms_triton = triton.testing.do_bench(lambda: flash_att_backward_split(q, k, v, o, do, lse_sum, NUM_DOCS, cuseq))
if HAS_FLEX:
    ms_flex = triton.testing.do_bench(bench_flex)

flops_bwd = 10 * NUM_DOCS * NUM_HEADS * (SEQ_LEN ** 2) * HEAD_DIM
tflops_flash = flops_bwd / (ms_flash * 1e-3) / 1e12
tflops_triton = flops_bwd / (ms_triton * 1e-3) / 1e12

print("\n--- Backward Pass Benchmark ---")
print(f"{'Implementation':<17} | {'Time (ms)':>10} | {'TFLOPs':>10}")
print("-" * 45)
print(f"{'SDPA FlashAttn-2':<17} | {ms_flash:>10.4f} | {tflops_flash:>10.1f}")
if HAS_FLEX:
    tflops_flex = flops_bwd / (ms_flex * 1e-3) / 1e12
    print(f"{'FlexAttention':<17} | {ms_flex:>10.4f} | {tflops_flex:>10.1f}")
print(f"{'Triton Custom':<17} | {ms_triton:>10.4f} | {tflops_triton:>10.1f}")

Generating Exact Mathematical Tensors...
Warming up SDPA...
Compiling FlexAttention...
Warming up Custom Triton...

--- Correctness Check (Max Absolute Diff against SDPA) ---
Flex vs SDPA -> dQ: 0.00781, dK: 0.01562, dV: 0.01562
Tri vs SDPA  -> dQ: 0.04688, dK: 0.04688, dV: 0.06250
*(Values below ~0.05 are considered numerically equivalent due to bf16 summation differences)*

--- Running Benchmarks ---

--- Backward Pass Benchmark ---
Implementation    |  Time (ms) |     TFLOPs
---------------------------------------------
SDPA FlashAttn-2  |     1.3211 |       97.5
FlexAttention     |     1.7971 |       71.7
Triton Custom     |     1.4519 |       88.7


In [31]:
import torch.nn.functional as F

print("\n--- Running Mathematical Correctness Check ---")

# 1. Run a Pure PyTorch Forward Pass (in FP32 for precision reference)
# Reshape packed inputs into (Batch, Heads, Seq_Len, Head_Dim)
q_pt = q.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).float().requires_grad_(True)
k_pt = k.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).float().requires_grad_(True)
v_pt = v.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).float().requires_grad_(True)
do_pt = do.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).float()

sm_scale = HEAD_DIM ** -0.5

# Manual Attention Math
scores = (q_pt @ k_pt.transpose(-2, -1)) * sm_scale
m = scores.max(dim=-1, keepdim=True).values
p = torch.exp(scores - m)
l_sum = p.sum(dim=-1, keepdim=True)
lse = m + torch.log(l_sum)
out = (p / l_sum) @ v_pt

# PyTorch Backward Pass
out.backward(do_pt)

# 2. Extract PyTorch Reference Gradients and cast back to Bfloat16
dq_ref = q_pt.grad.transpose(1, 2).reshape(NUM_DOCS * SEQ_LEN, NUM_HEADS, HEAD_DIM).to(DTYPE)
dk_ref = k_pt.grad.transpose(1, 2).reshape(NUM_DOCS * SEQ_LEN, NUM_HEADS, HEAD_DIM).to(DTYPE)
dv_ref = v_pt.grad.transpose(1, 2).reshape(NUM_DOCS * SEQ_LEN, NUM_HEADS, HEAD_DIM).to(DTYPE)

# 3. Format exact O and LSE for our Triton Kernel
# Triton FA expects LSE in Base-2, while PyTorch uses Base-e. We must scale it!
o_triton = out.transpose(1, 2).reshape(NUM_DOCS * SEQ_LEN, NUM_HEADS, HEAD_DIM).to(DTYPE)
lse_triton = (lse.squeeze(-1) * 1.44269504).transpose(0, 1).reshape(NUM_HEADS, NUM_DOCS * SEQ_LEN).to(DTYPE)

# 4. Run Triton Backward Split-Kernels
dq_tri, dk_tri, dv_tri = flash_att_backward_split(q, k, v, o_triton, do, lse_triton, NUM_DOCS)

# 5. Compare Maximum Absolute Error
err_dq = (dq_tri - dq_ref).abs().max().item()
err_dk = (dk_tri - dk_ref).abs().max().item()
err_dv = (dv_tri - dv_ref).abs().max().item()

ok_dq = torch.allclose(dq_tri, dq_ref, atol=3e-2, rtol=3e-2)
ok_dk = torch.allclose(dk_tri, dk_ref, atol=3e-2, rtol=3e-2)
ok_dv = torch.allclose(dv_tri, dv_ref, atol=3e-2, rtol=3e-2)

print(f"Max Error dQ: {err_dq:.4f} | OK: {ok_dq}")
print(f"Max Error dK: {err_dk:.4f} | OK: {ok_dk}")
print(f"Max Error dV: {err_dv:.4f} | OK: {ok_dv}")


--- Running Mathematical Correctness Check ---
Max Error dQ: 0.0156 | OK: True
Max Error dK: 0.0156 | OK: True
Max Error dV: 0.0078 | OK: True


In [72]:

#Assuming q,k,v to be of shape [num_tokens,num_heads,head_dim]
@triton.jit
def flash_att_backward_kernel(q_ptr,k_ptr,v_ptr,dq_ptr,dk_ptr,dv_ptr,do_ptr,lse_ptr,odo_sum_ptr,seq_len:tl.constexpr,sm_scale:tl.constexpr,
                       HEAD_DIM:tl.constexpr,BLOCK_M:tl.constexpr,BLOCK_N:tl.constexpr):
    
    pid=tl.program_id(axis=0) #which tokens in the doc-head
    pid_s=tl.program_id(axis=1) # which document
    pid_h=tl.program_id(axis=2) # which head in doc

    #stride between documents is num_heads*head_dim
    stride_tok=tl.num_programs(axis=2)*HEAD_DIM 
    stride_h=tl.num_programs(axis=1)*seq_len

    #no gqa assumptions here
    q_ptr_b=q_ptr+pid_s*stride_tok*seq_len + pid_h*HEAD_DIM #move pointers to the correspodning doc,head init
    k_ptr_b=k_ptr+pid_s*stride_tok*seq_len + pid_h*HEAD_DIM
    v_ptr_b=v_ptr+pid_s*stride_tok*seq_len + pid_h*HEAD_DIM
    dq_ptr_b=dq_ptr+pid_s*stride_tok*seq_len + pid_h*HEAD_DIM
    dk_ptr_b=dk_ptr+pid_s*stride_tok*seq_len + pid_h*HEAD_DIM
    dv_ptr_b=dv_ptr+pid_s*stride_tok*seq_len + pid_h*HEAD_DIM
    do_ptr_b=do_ptr+pid_s*stride_tok*seq_len + pid_h*HEAD_DIM
    
    # we will store this in form of (heads,docs) for mem coaelscing so different strides
    lse_ptr_b=lse_ptr+pid_h*stride_h + pid_s*seq_len
    odo_sum_ptr_b=odo_sum_ptr+pid_h*stride_h + pid_s*seq_len

    qk_scale = sm_scale * 1.44269504
    qk_scale=tl.cast(qk_scale,tl.bfloat16)

    #Load  block pointers for all required tensors in computation
    q_block_ptr=tl.make_block_ptr(q_ptr_b,shape=(seq_len,HEAD_DIM),strides=(stride_tok,1),
                                       offsets=(pid*BLOCK_M,0),block_shape=(BLOCK_M,HEAD_DIM),
                                    order=(1,0)) 
     
    k_block_ptr=tl.make_block_ptr(k_ptr_b,shape=(HEAD_DIM,seq_len),strides=(1,stride_tok),
                                offsets=(0,pid*BLOCK_N),block_shape=(HEAD_DIM,BLOCK_N),
                                order=(0,1)) 

    v_block_ptr=tl.make_block_ptr(v_ptr_b,shape=(seq_len,HEAD_DIM),strides=(stride_tok,1),
                                       offsets=(pid*BLOCK_N,0),block_shape=(BLOCK_N,HEAD_DIM),
                                    order=(1,0)) 
    
    do_block_ptr=tl.make_block_ptr(do_ptr_b,shape=(seq_len,HEAD_DIM),strides=(stride_tok,1),
                                offsets=(0,0),block_shape=(BLOCK_M,HEAD_DIM),
                                order=(1,0)) 
    
    dv_block_ptr=tl.make_block_ptr(dv_ptr_b,shape=(seq_len,HEAD_DIM),strides=(stride_tok,1),
                            offsets=(pid*BLOCK_N,0),block_shape=(BLOCK_N,HEAD_DIM),
                            order=(1,0)) 
    

    dk_block_ptr=tl.make_block_ptr(dk_ptr_b,shape=(seq_len,HEAD_DIM),strides=(stride_tok,1),
                            offsets=(pid*BLOCK_N,0),block_shape=(BLOCK_N,HEAD_DIM),
                            order=(1,0)) 
    # dk_block_ptr=tl.make_block_ptr(dk_ptr_b,shape=(HEAD_DIM,seq_len),strides=(1,stride_tok),
    #                             offsets=(0,0),block_shape=(HEAD_DIM,BLOCK_N),
    #                             order=(0,1)) 
    
    #strides 0
    odo_sum_block_ptr=tl.make_block_ptr(base=odo_sum_ptr_b,shape=(seq_len,),
                               strides=(1,),offsets=(pid*BLOCK_M,),
                               block_shape=(BLOCK_M,),order=(0,))
    
    lse_block_ptr=tl.make_block_ptr(base=lse_ptr_b,shape=(seq_len,),
                            strides=(1,),offsets=(pid*BLOCK_M,),
                            block_shape=(BLOCK_M,),order=(0,))
    
    #need to think about structure of this block ptr since withou head dim coalescing affected
    # thinking again block size M will acocount for coaelscing so need not worry 
    # wrong heads are issue for parallelization. better to head,seq format


    k=tl.load(k_block_ptr,boundary_check=(1,),padding_option="zero")
    v=tl.load(v_block_ptr,boundary_check=(0,),padding_option="zero")

    acc_dv=tl.zeros((BLOCK_N,HEAD_DIM),tl.float32)
    acc_dk=tl.zeros((BLOCK_N,HEAD_DIM),tl.float32)
    

    for i in range(0,seq_len,BLOCK_M):
        #laod do dq and q for calculating
        do=tl.load(do_block_ptr,boundary_check=(0,),padding_option="zero")
        q=tl.load(q_block_ptr,boundary_check=(0,),padding_option="zero")

        odo_sum=tl.load(odo_sum_block_ptr,boundary_check=(0,),padding_option="zero")
        lse=tl.load(lse_block_ptr,boundary_check=(0,),padding_option="zero")

        #calculate similarlity matrix sij=<qi,kj> shape is of B_MxB_N
        s=tl.dot(q,k)
        s=s*qk_scale
        #calculate prob via softmax here lse is log(exp(max)+Z),Z is sum of logits stored in forward
        p=tl.math.exp2(s-lse[:,None])
        
        #observe that oi=aijvj +k .This gives us dvj= sum_i aijdoi. A^Tdo
        acc_dv=tl.dot(tl.trans(p).to(tl.bfloat16),do,acc_dv)

        # To calcualate dL/daij we have <doi,dvj> 
        dp=tl.dot(do,tl.trans(v))

        # dL/dsij where sij=<qi,kj> . Using chain rule and total derivative we have dL/dsij =sum k dL/daik.daik/dsij
        # This is sumk dpij.pij(detlta(j,k)-pik)=pij(dpij-D) where D= <1,dO.O>
        ds=p*(dp.to(tl.bfloat16)-odo_sum[:,None]) #
        ds=ds.to(tl.bfloat16)
        #acc grad w.r.t q into dk
        acc_dk=tl.dot(tl.trans(ds),q*qk_scale,acc_dk)

        #accumulate grad w.r.t k into dq
            # ---- dq via atomic, raw pointers, fp32 contribution ----
        dq_contrib = tl.dot(ds, tl.trans(k)).to(tl.float32) * sm_scale
        offs_m  = i + tl.arange(0, BLOCK_M)
        dq_ptrs = dq_ptr_b + offs_m[:, None]*stride_tok + tl.arange(0, HEAD_DIM)[None, :]
        tl.atomic_add(dq_ptrs, dq_contrib, mask=(offs_m < seq_len)[:, None])
        
        #advance ptrs
        q_block_ptr=tl.advance(q_block_ptr,(BLOCK_M,0))
        odo_sum_block_ptr=tl.advance(odo_sum_block_ptr,(BLOCK_M,))
        lse_block_ptr=tl.advance(lse_block_ptr,(BLOCK_M,))

    acc_dk=acc_dk*sm_scale
    #store results
    tl.store(dk_block_ptr,acc_dk.to(tl.bfloat16), boundary_check=(0,))
    tl.store(dv_block_ptr,acc_dv.to(tl.bfloat16), boundary_check=(0,))

@triton.jit
def calc_odo_sum(o_ptr,do_ptr,odo_sum_ptr,seq_len,HEAD_DIM:tl.constexpr,BLOCK_K:tl.constexpr):
    pid=tl.program_id(axis=0)
    pid_h=tl.program_id(axis=2)
    pid_s=tl.program_id(axis=1)

    stride_tok=tl.num_programs(axis=2)*HEAD_DIM
    stride_h=tl.num_programs(axis=1)*seq_len

    o_ptr_b=o_ptr+pid_s*stride_tok*seq_len+pid_h*HEAD_DIM
    do_ptr_b=do_ptr+pid_s*stride_tok*seq_len+pid_h*HEAD_DIM

    odo_sum_ptr_b=odo_sum_ptr+pid_h*stride_h + pid_s*seq_len + pid*BLOCK_K + tl.arange(0,BLOCK_K)
    
    offset=pid_h*stride_h + pid_s*seq_len + pid*BLOCK_K + tl.arange(0,BLOCK_K)
    boundary=pid_h*stride_h + pid_s*seq_len + seq_len
    mask=offset<boundary

    o_block_ptr=tl.make_block_ptr(base=o_ptr_b,shape=(seq_len,HEAD_DIM),
                               strides=(stride_tok,1),offsets=(pid*BLOCK_K,0),
                               block_shape=(BLOCK_K,HEAD_DIM),order=(1,0))

    do_block_ptr=tl.make_block_ptr(base=do_ptr_b,shape=(seq_len,HEAD_DIM),
                               strides=(stride_tok,1),offsets=(pid*BLOCK_K,0),
                               block_shape=(BLOCK_K,HEAD_DIM),order=(1,0))

    
    o=tl.load(o_block_ptr,boundary_check=(0,),padding_option="zero")
    do=tl.load(do_block_ptr,boundary_check=(0,),padding_option="zero")

    odo=tl.sum(o*do,axis=1)

    tl.store(odo_sum_ptr_b,odo,mask=mask)


def flash_att_backward(q,k,v,o,do,lse_sum,num_docs,BLOCK_M=32,BLOCK_N=64,BLOCK_K=512):
    cum_seq_len,heads,head_dim=q.shape
    seq_len=cum_seq_len//num_docs

    dq,dk,dv,odo_sum=torch.zeros_like(q),torch.empty_like(k),torch.empty_like(v),torch.empty_like(lse_sum)
    grid1=lambda meta:(triton.cdiv(seq_len,BLOCK_K),num_docs,heads)
    grid2=lambda meta:(triton.cdiv(seq_len,BLOCK_N),num_docs,heads)


    calc_odo_sum[grid1](o,do,odo_sum,seq_len,head_dim,BLOCK_K=BLOCK_K,num_warps=8,num_stages=4)
    sm_scale=1/(head_dim**0.5)
    
    # print(odo_sum.dtype,odo_sum.shape,torch.sum(o*do,axis=-1).permute(1,0),odo_sum, (o.float() * do.float()).sum(-1).permute(1,0))
    flash_att_backward_kernel[grid2](q,k,v,dq,dk,dv,do,lse_sum,odo_sum,
                                 seq_len,sm_scale,head_dim,BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N,
        num_warps=8, num_stages=3)

    return dq,dk,dv


SEQ_LEN=1024
NUM_DOCS=8
NUM_HEADS=12
HEAD_DIM=128
DTYPE=torch.bfloat16

q = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
k = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
v = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
o = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
do = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
lse_sum=torch.randn(NUM_HEADS,NUM_DOCS*SEQ_LEN,device=DEVICE, dtype=DTYPE)

flash_att_backward(q,k,v,o,do,lse_sum,NUM_DOCS)
ms_triton = triton.testing.do_bench(lambda: flash_att_backward(q,k,v,o,do,lse_sum,NUM_DOCS)
)
ms_triton


9.51945251888699

In [86]:
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

print("Preparing PyTorch Baseline...")

# Reshape packed inputs into (Batch, Heads, Seq_Len, Head_Dim) for PyTorch SDPA
q_ref = q.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
k_ref = k.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
v_ref = v.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
do_ref = do.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone()

# Explicitly force the FlashAttention backend (this will throw an error if your hardware/shapes don't support it, 
# guaranteeing it won't silently fall back to slow Math)
backend = SDPBackend.FLASH_ATTENTION
print(f"Enforcing PyTorch Backend: {backend.name}")

with sdpa_kernel(backend):
    # Run a single forward pass to build the autograd graph
    out_ref = F.scaled_dot_product_attention(q_ref, k_ref, v_ref)

def bench_sdpa_backward():
    # Clear grads to avoid accumulation overhead during benchmarking
    q_ref.grad = None
    k_ref.grad = None
    v_ref.grad = None
    
    # The backward pass inherits the backend chosen during the forward pass graph construction
    out_ref.backward(do_ref, retain_graph=True)

# 1. Benchmark PyTorch FlashAttention-2
ms_flash = triton.testing.do_bench(bench_sdpa_backward)

# 2. Benchmark your Custom Triton Kernel
ms_triton = triton.testing.do_bench(lambda: flash_att_backward(q, k, v, o, do, lse_sum, NUM_DOCS))

# Approx FLOPs for backward pass: 10 * B * H * S^2 * D
flops_bwd = 10 * NUM_DOCS * NUM_HEADS * (SEQ_LEN ** 2) * HEAD_DIM
tflops_flash = flops_bwd / (ms_flash * 1e-3) / 1e12
tflops_triton = flops_bwd / (ms_triton * 1e-3) / 1e12

print("\n--- Backward Pass Benchmark ---")
print(f"{'Implementation':<15} | {'Time (ms)':>10} | {'TFLOPs':>10}")
print("-" * 42)
print(f"{'FlashAttention':<15} | {ms_flash:>10.4f} | {tflops_flash:>10.1f}")
print(f"{'Triton Custom':<15} | {ms_triton:>10.4f} | {tflops_triton:>10.1f}")
print(f"\nFlashAttention is {ms_triton / ms_flash:.2f}x faster.")

Preparing PyTorch Baseline...
Enforcing PyTorch Backend: FLASH_ATTENTION

--- Backward Pass Benchmark ---
Implementation  |  Time (ms) |     TFLOPs
------------------------------------------
FlashAttention  |     1.5705 |       82.0
Triton Custom   |     8.5902 |       15.0

FlashAttention is 5.47x faster.


In [91]:
import torch
import triton
import triton.language as tl
from torch.nn.attention import sdpa_kernel, SDPBackend
import torch.nn.functional as F

DEVICE = "cuda:2"
DTYPE = torch.bfloat16

# -----------------------------------------------------------
# 1. KERNEL 1: Compute dQ (Anchors Q, loops over K/V)
# -----------------------------------------------------------
@triton.autotune(
    configs=[
        # A6000 strict SRAM limit configs (< 99KB)
        # 1. Balanced blocks, low pipeline
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 64}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 32}, num_warps=4, num_stages=2),
        
        # 2. Maximum square block, NO pipelining (1 stage only)
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 64}, num_warps=8, num_stages=1), 
        
        # 3. Small square, deeper pipeline for latency hiding
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 32}, num_warps=4, num_stages=3),
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 32}, num_warps=8, num_stages=3),
        
        # 4. Extreme skew for edge cases
        triton.Config({'BLOCK_M': 16, 'BLOCK_N': 64}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 16}, num_warps=4, num_stages=2),
    ],
    key=['seq_len', 'HEAD_DIM'],
)
@triton.jit
def bwd_kernel_dq(
    q_ptr, k_ptr, v_ptr, do_ptr, dq_ptr, lse_ptr, odo_sum_ptr,
    seq_len: tl.constexpr, sm_scale: tl.constexpr, HEAD_DIM: tl.constexpr, 
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr
):
    pid_m = tl.program_id(0) # Q block index
    pid_s = tl.program_id(1)
    pid_h = tl.program_id(2)

    stride_tok = tl.num_programs(2) * HEAD_DIM 
    stride_h = tl.num_programs(1) * seq_len
    base_offset = pid_s * stride_tok * seq_len + pid_h * HEAD_DIM

    # Anchor Q, dO, dQ
    q_block = tl.make_block_ptr(base=q_ptr + base_offset, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(pid_m * BLOCK_M, 0), block_shape=(BLOCK_M, HEAD_DIM), order=(1, 0))
    do_block = tl.make_block_ptr(base=do_ptr + base_offset, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(pid_m * BLOCK_M, 0), block_shape=(BLOCK_M, HEAD_DIM), order=(1, 0))
    dq_block = tl.make_block_ptr(base=dq_ptr + base_offset, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(pid_m * BLOCK_M, 0), block_shape=(BLOCK_M, HEAD_DIM), order=(1, 0))
    
    lse_block = tl.make_block_ptr(base=lse_ptr + pid_h * stride_h + pid_s * seq_len, shape=(seq_len,), strides=(1,), offsets=(pid_m * BLOCK_M,), block_shape=(BLOCK_M,), order=(0,))
    odo_block = tl.make_block_ptr(base=odo_sum_ptr + pid_h * stride_h + pid_s * seq_len, shape=(seq_len,), strides=(1,), offsets=(pid_m * BLOCK_M,), block_shape=(BLOCK_M,), order=(0,))

    # Sliding K and V
    k_block = tl.make_block_ptr(base=k_ptr + base_offset, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(0, 0), block_shape=(BLOCK_N, HEAD_DIM), order=(1, 0))
    v_block = tl.make_block_ptr(base=v_ptr + base_offset, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(0, 0), block_shape=(BLOCK_N, HEAD_DIM), order=(1, 0))

    q = tl.load(q_block, boundary_check=(0,), padding_option="zero")
    do = tl.load(do_block, boundary_check=(0,), padding_option="zero")
    lse = tl.load(lse_block, boundary_check=(0,), padding_option="zero")
    odo = tl.load(odo_block, boundary_check=(0,), padding_option="zero")

    acc_dq = tl.zeros((BLOCK_M, HEAD_DIM), tl.float32)
    qk_scale = sm_scale * 1.44269504

    for n in range(0, seq_len, BLOCK_N):
        k = tl.load(k_block, boundary_check=(0,), padding_option="zero")
        v = tl.load(v_block, boundary_check=(0,), padding_option="zero")

        s = tl.dot(q, tl.trans(k)) * qk_scale
        p = tl.math.exp2(s - lse[:, None])
        
        dp = tl.dot(do, tl.trans(v), out_dtype=tl.float32)
        ds = p * (dp - odo[:, None])
        
        acc_dq += tl.dot(ds.to(tl.bfloat16), k, out_dtype=tl.float32)

        k_block = tl.advance(k_block, (BLOCK_N, 0))
        v_block = tl.advance(v_block, (BLOCK_N, 0))

    acc_dq = acc_dq * sm_scale
    tl.store(dq_block, acc_dq.to(tl.bfloat16), boundary_check=(0,))

# -----------------------------------------------------------
# 2. KERNEL 2: Compute dK, dV (Anchors K/V, loops over Q)
# -----------------------------------------------------------
@triton.autotune(
    configs=[
        # A6000 strict SRAM limit configs (< 99KB)
        # 1. Balanced blocks, low pipeline
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 64}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 32}, num_warps=4, num_stages=2),
        
        # 2. Maximum square block, NO pipelining (1 stage only)
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 64}, num_warps=8, num_stages=1), 
        
        # 3. Small square, deeper pipeline for latency hiding
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 32}, num_warps=4, num_stages=3),
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 32}, num_warps=8, num_stages=3),
        
        # 4. Extreme skew for edge cases
        triton.Config({'BLOCK_M': 16, 'BLOCK_N': 64}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 16}, num_warps=4, num_stages=2),
    ],
    key=['seq_len', 'HEAD_DIM'],
)
@triton.jit
def bwd_kernel_dk_dv(
    q_ptr, k_ptr, v_ptr, do_ptr, dk_ptr, dv_ptr, lse_ptr, odo_sum_ptr,
    seq_len: tl.constexpr, sm_scale: tl.constexpr, HEAD_DIM: tl.constexpr, 
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr
):
    pid_n = tl.program_id(0) # KV block index
    pid_s = tl.program_id(1)
    pid_h = tl.program_id(2)

    stride_tok = tl.num_programs(2) * HEAD_DIM 
    stride_h = tl.num_programs(1) * seq_len
    base_offset = pid_s * stride_tok * seq_len + pid_h * HEAD_DIM

    # Anchor K, V, dK, dV
    k_block = tl.make_block_ptr(base=k_ptr + base_offset, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(pid_n * BLOCK_N, 0), block_shape=(BLOCK_N, HEAD_DIM), order=(1, 0))
    v_block = tl.make_block_ptr(base=v_ptr + base_offset, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(pid_n * BLOCK_N, 0), block_shape=(BLOCK_N, HEAD_DIM), order=(1, 0))
    dk_block = tl.make_block_ptr(base=dk_ptr + base_offset, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(pid_n * BLOCK_N, 0), block_shape=(BLOCK_N, HEAD_DIM), order=(1, 0))
    dv_block = tl.make_block_ptr(base=dv_ptr + base_offset, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(pid_n * BLOCK_N, 0), block_shape=(BLOCK_N, HEAD_DIM), order=(1, 0))

    # Sliding Q and dO
    q_block = tl.make_block_ptr(base=q_ptr + base_offset, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(0, 0), block_shape=(BLOCK_M, HEAD_DIM), order=(1, 0))
    do_block = tl.make_block_ptr(base=do_ptr + base_offset, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(0, 0), block_shape=(BLOCK_M, HEAD_DIM), order=(1, 0))
    
    lse_block = tl.make_block_ptr(base=lse_ptr + pid_h * stride_h + pid_s * seq_len, shape=(seq_len,), strides=(1,), offsets=(0,), block_shape=(BLOCK_M,), order=(0,))
    odo_block = tl.make_block_ptr(base=odo_sum_ptr + pid_h * stride_h + pid_s * seq_len, shape=(seq_len,), strides=(1,), offsets=(0,), block_shape=(BLOCK_M,), order=(0,))

    k = tl.load(k_block, boundary_check=(0,), padding_option="zero")
    v = tl.load(v_block, boundary_check=(0,), padding_option="zero")

    acc_dk = tl.zeros((BLOCK_N, HEAD_DIM), tl.float32)
    acc_dv = tl.zeros((BLOCK_N, HEAD_DIM), tl.float32)
    qk_scale = sm_scale * 1.44269504

    for m in range(0, seq_len, BLOCK_M):
        q = tl.load(q_block, boundary_check=(0,), padding_option="zero")
        do = tl.load(do_block, boundary_check=(0,), padding_option="zero")
        lse = tl.load(lse_block, boundary_check=(0,), padding_option="zero")
        odo = tl.load(odo_block, boundary_check=(0,), padding_option="zero")

        s = tl.dot(q, tl.trans(k)) * qk_scale
        p = tl.math.exp2(s - lse[:, None])
        
        dp = tl.dot(do, tl.trans(v), out_dtype=tl.float32)
        ds = p * (dp - odo[:, None])

        acc_dv += tl.dot(tl.trans(p).to(tl.bfloat16), do, out_dtype=tl.float32)
        acc_dk += tl.dot(tl.trans(ds).to(tl.bfloat16), q, out_dtype=tl.float32)

        q_block = tl.advance(q_block, (BLOCK_M, 0))
        do_block = tl.advance(do_block, (BLOCK_M, 0))
        lse_block = tl.advance(lse_block, (BLOCK_M,))
        odo_block = tl.advance(odo_block, (BLOCK_M,))

    acc_dk = acc_dk * sm_scale
    tl.store(dk_block, acc_dk.to(tl.bfloat16), boundary_check=(0,))
    tl.store(dv_block, acc_dv.to(tl.bfloat16), boundary_check=(0,))

# -----------------------------------------------------------
# Precompute Delta & Wrapper
# -----------------------------------------------------------
@triton.jit
def calc_odo_sum(o_ptr, do_ptr, odo_sum_ptr, seq_len, HEAD_DIM: tl.constexpr, BLOCK_K: tl.constexpr):
    pid = tl.program_id(axis=0)
    pid_s = tl.program_id(axis=1)
    pid_h = tl.program_id(axis=2)

    stride_tok = tl.num_programs(axis=2) * HEAD_DIM
    stride_h = tl.num_programs(axis=1) * seq_len
    o_ptr_b = o_ptr + pid_s * stride_tok * seq_len + pid_h * HEAD_DIM
    do_ptr_b = do_ptr + pid_s * stride_tok * seq_len + pid_h * HEAD_DIM
    
    offset = pid_h * stride_h + pid_s * seq_len + pid * BLOCK_K + tl.arange(0, BLOCK_K)
    mask = offset < (pid_h * stride_h + pid_s * seq_len + seq_len)

    o_block = tl.make_block_ptr(base=o_ptr_b, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(pid * BLOCK_K, 0), block_shape=(BLOCK_K, HEAD_DIM), order=(1, 0))
    do_block = tl.make_block_ptr(base=do_ptr_b, shape=(seq_len, HEAD_DIM), strides=(stride_tok, 1), offsets=(pid * BLOCK_K, 0), block_shape=(BLOCK_K, HEAD_DIM), order=(1, 0))

    o = tl.load(o_block, boundary_check=(0,), padding_option="zero")
    do = tl.load(do_block, boundary_check=(0,), padding_option="zero")

    odo = tl.sum(o.to(tl.float32) * do.to(tl.float32), axis=1)
    tl.store(odo_sum_ptr + offset, odo, mask=mask)

def flash_att_backward_split(q, k, v, o, do, lse_sum, num_docs):
    cum_seq_len, heads, head_dim = q.shape
    seq_len = cum_seq_len // num_docs

    dq = torch.empty_like(q)
    dk = torch.empty_like(k)
    dv = torch.empty_like(v)
    odo_sum = torch.empty_like(lse_sum)
    sm_scale = 1 / (head_dim ** 0.5)

    # 1. Delta pre-compute
    calc_odo_sum[(triton.cdiv(seq_len, 256), num_docs, heads)](o, do, odo_sum, seq_len, head_dim, BLOCK_K=256)
    
    # 2. Split Kernels (Launched asynchronously)
    grid_dq = lambda META: (triton.cdiv(seq_len, META['BLOCK_M']), num_docs, heads)
    grid_dkdv = lambda META: (triton.cdiv(seq_len, META['BLOCK_N']), num_docs, heads)

    bwd_kernel_dq[grid_dq](q, k, v, do, dq, lse_sum, odo_sum, seq_len, sm_scale, head_dim)
    bwd_kernel_dk_dv[grid_dkdv](q, k, v, do, dk, dv, lse_sum, odo_sum, seq_len, sm_scale, head_dim)

    return dq, dk, dv

# -----------------------------------------------------------
# Benchmark
# -----------------------------------------------------------
SEQ_LEN = 1024
NUM_DOCS = 8
NUM_HEADS = 12
HEAD_DIM = 128

q = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
k = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
v = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
o = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
do = torch.randn(NUM_DOCS*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
lse_sum = torch.randn(NUM_HEADS, NUM_DOCS*SEQ_LEN, device=DEVICE, dtype=DTYPE)

# Baseline prep
q_ref = q.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
k_ref = k.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
v_ref = v.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone().requires_grad_(True)
do_ref = do.view(NUM_DOCS, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2).detach().clone()

with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
    out_ref = F.scaled_dot_product_attention(q_ref, k_ref, v_ref)

def bench_sdpa():
    q_ref.grad = None
    k_ref.grad = None
    v_ref.grad = None
    out_ref.backward(do_ref, retain_graph=True)

# Warmup JIT
_ = flash_att_backward_split(q, k, v, o, do, lse_sum, NUM_DOCS)

ms_flash = triton.testing.do_bench(bench_sdpa)
ms_triton = triton.testing.do_bench(lambda: flash_att_backward_split(q, k, v, o, do, lse_sum, NUM_DOCS))

flops_bwd = 10 * NUM_DOCS * NUM_HEADS * (SEQ_LEN ** 2) * HEAD_DIM
tflops_flash = flops_bwd / (ms_flash * 1e-3) / 1e12
tflops_triton = flops_bwd / (ms_triton * 1e-3) / 1e12

print("\n--- Split-Kernel FA2 Backward Benchmark ---")
print(f"{'Implementation':<15} | {'Time (ms)':>10} | {'TFLOPs':>10}")
print("-" * 42)
print(f"{'FlashAttention2':<15} | {ms_flash:>10.4f} | {tflops_flash:>10.1f}")
print(f"{'Triton FA2 Split':<15} | {ms_triton:>10.4f} | {tflops_triton:>10.1f}")


--- Split-Kernel FA2 Backward Benchmark ---
Implementation  |  Time (ms) |     TFLOPs
------------------------------------------
FlashAttention2 |     1.5730 |       81.9
Triton FA2 Split |     2.0514 |       62.8


In [88]:
best = bwd_kernel_dq.best_config

print("Block Sizes:", best.kwargs)
print("Num Warps:  ", best.num_warps)
print("Num Stages: ", best.num_stages)

Block Sizes: {'BLOCK_M': 64, 'BLOCK_N': 32}
Num Warps:   4
Num Stages:  2


In [ ]:
@triton.autotune(
    configs=[
        triton.Config({'BLOCK_M': 128, 'BLOCK_N': 64}, num_warps=8, num_stages=3),
        triton.Config({'BLOCK_M': 128, 'BLOCK_N': 64}, num_warps=4, num_stages=3),
        triton.Config({'BLOCK_M': 64,  'BLOCK_N': 64}, num_warps=4, num_stages=3),
        triton.Config({'BLOCK_M': 64,  'BLOCK_N': 64}, num_warps=8, num_stages=3),
        triton.Config({'BLOCK_M': 128, 'BLOCK_N': 32}, num_warps=8, num_stages=3),
    ],
    key=['doc_len', 'HEAD_DIM'],
)
@triton.jit
def flash_att_intra_doc_masked(q_ptr,k_ptr,v_ptr,o_ptr,cuseq_ptr,num_doc,
                     sm_scale,HEAD_DIM:tl.constexpr,BLOCK_M:tl.constexpr,BLOCK_N:tl.constexpr):
    
    #Read grid
    pid=tl.program_id(axis=0)
    pid_doc=tl.program_id(axis=1)
    pid_h=tl.program_id(axis=2)


    #Assume data in shape (num_tokens,num_head,head_dim)
    #Load the doc start and end idx and calculate doc length using them

    start=tl.load(cuseq_ptr+pid_doc)
    end=tl.load(cuseq_ptr+pid_doc+1)

    doc_len=end-start

    #Load q,k,v ptrs
    stride_tok=tl.num_programs(axis=2) * HEAD_DIM 

    q_ptr_b=q_ptr+start*stride_tok+ pid_h*HEAD_DIM 
    k_ptr_b=k_ptr+start*stride_tok + pid_h*HEAD_DIM  
    v_ptr_b=v_ptr+start*stride_tok + pid_h*HEAD_DIM 
    o_ptr_b=o_ptr+start*stride_tok+ pid_h*HEAD_DIM 

    q_block_ptr=tl.make_block_ptr(base=q_ptr_b,
                                  shape=(doc_len,HEAD_DIM),
                                  strides=(stride_tok,1),
                                  offsets=(pid*BLOCK_M,0),
                                  block_shape=(BLOCK_M,HEAD_DIM),
                                  order=(1,0))
    #Load k transposed
    k_block_ptr=tl.make_block_ptr(base=k_ptr_b,
                                  shape=(HEAD_DIM,doc_len),
                                  strides=(1,stride_tok),
                                  offsets=(0,0),
                                  block_shape=(HEAD_DIM,BLOCK_N),
                                  order=(0,1))
    
    v_block_ptr=tl.make_block_ptr(base=v_ptr_b,
                                shape=(doc_len,HEAD_DIM),
                                strides=(stride_tok,1),
                                offsets=(0,0),
                                block_shape=(BLOCK_N,HEAD_DIM),
                                order=(1,0))
    
    o_block_ptr=tl.make_block_ptr(base=o_ptr_b,
                                  shape=(doc_len,HEAD_DIM),
                                  strides=(stride_tok,1),
                                  offsets=(pid*BLOCK_M,0),
                                  block_shape=(BLOCK_M,HEAD_DIM),
                                  order=(1,0))
    
	

    #Some small optimization on scale claude suggested to use t.math.exp which has hardware support
    qk_scale = sm_scale * 1.44269504
    q = tl.load(q_block_ptr,boundary_check=(0,),padding_option="zero")
    q = (q * qk_scale).to(tl.bfloat16)
    
    #Buffers to accumulate res ,max and sums
    acc=tl.zeros((BLOCK_M,HEAD_DIM),tl.float32)
    max_val=tl.full((BLOCK_M,),float("-inf"),tl.float32)
    sum_val=tl.zeros((BLOCK_M,),tl.float32)

    q_offset=pid * BLOCK_M + tl.arange(0, BLOCK_M)
    doc_bias=tl.where(q_offset<doc_len,0.0,-10000.0) # offsets for  doc masking which q crossing boundary


    #safe part no causality got till start of query block
    for i in range(0,pid*BLOCK_M,BLOCK_N): #There will be some tail  q # We can assume that q block size will be greater than kv

        k=tl.load(k_block_ptr,boundary_check=(1,),padding_option="zero")
        v=tl.load(v_block_ptr,boundary_check=(0,),padding_option="zero")

        scores=tl.dot(q,k) + doc_bias[:,None]
        new_max=tl.maximum(max_val,tl.max(scores,axis=1))
        scores_exp=tl.math.exp2(scores-new_max[:,None])
        update_factor=tl.math.exp2(max_val-new_max)

        acc=acc*update_factor[:,None]

        acc=tl.dot(scores_exp.to(tl.bfloat16),v,acc)

        sum_val=sum_val*update_factor + tl.sum(scores_exp,axis=1)
        max_val=new_max
        
        k_block_ptr=tl.advance(k_block_ptr,(0,BLOCK_N))
        v_block_ptr=tl.advance(v_block_ptr,(BLOCK_N,0))


    #Causal and doc masking(Need to do block masking padding for q,k,v)

    limit_causal = tl.minimum((pid + 1) * BLOCK_M, doc_len)
    for i in range(pid*BLOCK_M,limit_causal,BLOCK_N): #We ensure that BLOCK_N divides BLOCK_M so no tailwind,(Offcourse doc masking remains)
        k=tl.load(k_block_ptr,boundary_check=(1,),padding_option="zero")
        v=tl.load(v_block_ptr,boundary_check=(0,),padding_option="zero")

        scores=tl.dot(q,k)
        k_offset=i+tl.arange(0,BLOCK_N)

        causal_bias=tl.where(q_offset[:, None] >= k_offset[None, :], 0.0, -10000.0) 
        #This creates an M,N causal mask similar to float inf mask in standard torch inf
        scores=scores + causal_bias + doc_bias[:,None]       

        new_max=tl.maximum(max_val,tl.max(scores,axis=1))
        scores_exp=tl.math.exp2(scores-new_max[:,None])
        update_factor=tl.math.exp2(max_val-new_max)

        acc=acc*update_factor[:,None]
        acc=tl.dot(scores_exp.to(tl.bfloat16),v,acc)

        sum_val=sum_val*update_factor + tl.sum(scores_exp,axis=1)
        max_val=new_max

        k_block_ptr=tl.advance(k_block_ptr,(0,BLOCK_N))
        v_block_ptr=tl.advance(v_block_ptr,(BLOCK_N,0))

    acc = acc / sum_val[:, None]
    tl.store(o_block_ptr, acc.to(tl.bfloat16), boundary_check=(0,))
    
    return


def triton_intradoc_attention(q,k,v,cuseq):
    out=torch.empty_like(q)
    num_docs=cuseq.shape[0]-1
    qn,_,_=q.shape
    # print(q.shape,num_docs)
    
    seq_lens = cuseq[1:] - cuseq[:-1]
    max_len = seq_lens.max().item()
    # BLOCK_M,BLOCK_N=128,32
    sm_scale=1/(HEAD_DIM**0.5)
    
    grid=lambda META: (triton.cdiv(max_len,META["BLOCK_M"]),num_docs,NUM_HEADS)
    flash_att_intra_doc_masked[grid](q,k,v,out,cuseq,num_docs,sm_scale,HEAD_DIM)
    return out



compiled_flex = torch.compile(flex_attention, dynamic=False)

def setup_flex_attention(q, k, v, doc_ids):
    q_f = q.unsqueeze(0).transpose(1, 2) 
    k_f = k.unsqueeze(0).transpose(1, 2)
    v_f = v.unsqueeze(0).transpose(1, 2)
    
    def doc_causal_mask(b, h, q_idx, kv_idx):
        return (q_idx >= kv_idx) & (doc_ids[q_idx] == doc_ids[kv_idx])

    total_seq = q_f.shape[2]
    block_mask = create_block_mask(doc_causal_mask, B=1, H=1, Q_LEN=total_seq, KV_LEN=total_seq, device=DEVICE)

    def run_flex():
        return compiled_flex(q_f, k_f, v_f, block_mask=block_mask)
        
    # FORCE multiple warmup passes to ensure Dynamo finishes tracing
    print("Warming up FlexAttention (Tracing + Compilation)...")
    for _ in range(3):
        _ = run_flex()
    print("Warmup Complete.")
    
    return run_flex

# -------------------------------------------------------------------------
# 3. BENCHMARK HARNESS
# -------------------------------------------------------------------------
# -------------------------------------------------------------------------
# 3. REAL-WORLD VARIABLE LENGTH BENCHMARK HARNESS
# -------------------------------------------------------------------------
def run_variable_benchmark(num_docs):
    # 1. Generate realistic jagged document lengths between 128 and 1024
    lengths = torch.randint(low=256, high=1025, size=(num_docs,), dtype=torch.int32, device=DEVICE)
    total_tokens = lengths.sum().item()
    max_len = lengths.max().item()
    min_len = lengths.min().item()

    # 2. Build cu_seq_lens
    cu_seq_lens = torch.zeros(num_docs + 1, dtype=torch.int32, device=DEVICE)
    cu_seq_lens[1:] = torch.cumsum(lengths, dim=0)

    # 3. Build doc_ids for FlexAttention
    doc_ids = torch.zeros(total_tokens, dtype=torch.int32, device=DEVICE)
    for i in range(num_docs):
        doc_ids[cu_seq_lens[i]:cu_seq_lens[i+1]] = i
        
    # 4. Generate packed Q, K, V
    q = torch.randn(total_tokens, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
    k = torch.randn(total_tokens, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
    v = torch.randn(total_tokens, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)

    # 5. Setup & Warmup
    _ = triton_intradoc_attention(q, k, v, cu_seq_lens)
    run_flex = setup_flex_attention(q, k, v, doc_ids)

    # 6. Correctness verification
    out_triton = triton_intradoc_attention(q, k, v, cu_seq_lens)
    out_flex_raw = run_flex()
    out_flex = out_flex_raw.transpose(1, 2).squeeze(0)

    max_err = (out_triton - out_flex).abs().max().item()
    ok = torch.allclose(out_triton, out_flex, atol=2e-2, rtol=2e-2)

    # 7. Benchmarking Iterations
    ms_triton = triton.testing.do_bench(lambda: triton_intradoc_attention(q, k, v, cu_seq_lens))
    ms_flex = triton.testing.do_bench(lambda: run_flex())

    # 8. Calculate Exact Causal FLOPs for the jagged lengths
    flops = 0
    for i in range(num_docs):
        seq = lengths[i].item()
        flops += 2 * NUM_HEADS * (seq ** 2) * HEAD_DIM
        
    tflops_triton = flops / (ms_triton * 1e-3) / 1e12
    tflops_flex = flops / (ms_flex * 1e-3) / 1e12

    return total_tokens, min_len, max_len, ms_triton, ms_flex, tflops_triton, tflops_flex, max_err, ok


print(f"Benchmarking Packed Causal Attention with Variable Doc Lengths (128-1024)")
print(f"{'Docs':>4} | {'TotalToks':>9} {'Min-Max':>10} | {'Triton(ms)':>10} {'Flex(ms)':>9} | {'Tri TFLOPs':>10} {'Flex TFLOPs':>11} | {'Ratio':>6} {'MaxErr':>8} {'OK':>4}")
print("-" * 105)

# Test from 4 documents up to 64 densely packed documents
for num_docs in [8,16]:
    tot, min_l, max_l, mt, mf, tt, tf, err, ok = run_variable_benchmark(num_docs)
    
    length_str = f"{min_l}-{max_l}"
    ratio = mt / mf if mf > 0 else 0
    print(f"{num_docs:>4} | {tot:>9} {length_str:>10} | {mt:>10.4f} {mf:>9.4f} | {tt:>10.1f} {tf:>11.1f} | {ratio:>6.2f} {err:>8.4f} {str(ok):>4}")

Benchmarking Packed Causal Attention with Variable Doc Lengths (128-1024)
Docs | TotalToks    Min-Max | Triton(ms)  Flex(ms) | Tri TFLOPs Flex TFLOPs |  Ratio   MaxErr   OK
---------------------------------------------------------------------------------------------------------
Warming up FlexAttention (Tracing + Compilation)...


InductorError: RuntimeError: No valid triton configs. OutOfMemoryError: out of resource: triton_tem_fused_0 Required: 106496 Hardware limit:101376 Reducing block sizes or `num_stages` may help.


In [ ]:

#Assuming q,k,v to be of shape [num_tokens,num_heads,head_dim]
@triton.jit
def flash_att_backward_kernel(q_ptr,k_ptr,v_ptr,dq_ptr,dk_ptr,dv_ptr,do_ptr,lse_ptr,odo_sum_ptr,seq_len:tl.constexpr,sm_scale:tl.constexpr,
                       HEAD_DIM:tl.constexpr,BLOCK_M:tl.constexpr,BLOCK_N:tl.constexpr):
    
	pid=tl.program_id(axis=0) #which tokens in the doc-head
	pid_s=tl.program_id(axis=1) # which document
	pid_h=tl.program_id(axis=2) # which head in doc

	#stride between documents is num_heads*head_dim
	stride_tok=tl.num_programs(axis=2)*HEAD_DIM 
	stride_h=tl.num_programs(axis=1)*seq_len

	#no gqa assumptions here
	q_ptr_b=q_ptr+pid_s*stride_tok + pid_h*HEAD_DIM #move pointers to the correspodning doc,head init
	k_ptr_b=k_ptr+pid_s*stride_tok + pid_h*HEAD_DIM
	v_ptr_b=v_ptr+pid_s*stride_tok + pid_h*HEAD_DIM
	dq_ptr_b=dq_ptr+pid_s*stride_tok + pid_h*HEAD_DIM
	dk_ptr_b=dk_ptr+pid_s*stride_tok + pid_h*HEAD_DIM
	dv_ptr_b=dv_ptr+pid_s*stride_tok + pid_h*HEAD_DIM
	do_ptr_b=do_ptr+pid_s*stride_tok + pid_h*HEAD_DIM
	
	# we will store this in form of (heads,docs) for mem coaelscing so different strides
	lse_ptr_b=lse_ptr+pid_h*stride_h + pid_s*seq_len
	odo_sum_ptr_b=odo_sum_ptr+pid_h*stride_h + pid_s*seq_len




	#Load  block pointers for all required tensors in computation
	q_block_ptr=tl.make_block_ptr(q_ptr_b,shape=(seq_len,HEAD_DIM),strides=(stride_tok,0),
							   		offsets=(pid*BLOCK_M,0),block_shape=(BLOCK_M,HEAD_DIM),
									order=(1,0)) 
	 
	k_block_ptr=tl.make_block_ptr(k_ptr_b,shape=(HEAD_DIM,seq_len),strides=(0,stride_tok),
								offsets=(0,0),block_shape=(HEAD_DIM,BLOCK_N),
								order=(0,1)) 

	v_block_ptr=tl.make_block_ptr(v_ptr_b,shape=(seq_len,HEAD_DIM),strides=(stride_tok,0),
							   		offsets=(pid*BLOCK_N,0),block_shape=(BLOCK_N,HEAD_DIM),
									order=(1,0)) 
	
	dq_block_ptr=tl.make_block_ptr(dq_ptr_b,shape=(seq_len,HEAD_DIM),strides=(stride_tok,0),
								offsets=(pid*BLOCK_M,0),block_shape=(BLOCK_M,HEAD_DIM),
								order=(1,0)) 
	
	do_block_ptr=tl.make_block_ptr(do_ptr_b,shape=(seq_len,HEAD_DIM),strides=(stride_tok,0),
								offsets=(pid*BLOCK_M,0),block_shape=(BLOCK_M,HEAD_DIM),
								order=(1,0)) 
	
	dv_block_ptr=tl.make_block_ptr(dv_ptr_b,shape=(seq_len,HEAD_DIM),strides=(stride_tok,0),
							offsets=(pid*BLOCK_N,0),block_shape=(BLOCK_N,HEAD_DIM),
							order=(1,0)) 
	
	dk_block_ptr=tl.make_block_ptr(dk_ptr_b,shape=(HEAD_DIM,seq_len),strides=(0,stride_tok),
								offsets=(0,0),block_shape=(HEAD_DIM,BLOCK_N),
								order=(0,1)) 
	
	#strides 0
	odo_sum_block_ptr=tl.make_block_ptr(base=odo_sum_ptr_b,shape=(seq_len,),
							   strides=(0,),offsets=(pid*BLOCK_M,),
							   boshape=(BLOCK_M,),order=(0,))
	
	lse_block_ptr=tl.make_block_ptr(base=lse_ptr_b,shape=(seq_len,),
							strides=(0,),offsets=(pid*BLOCK_M,),
							shape=(BLOCK_M,),order=(0,))
	
	#need to think about structure of this block ptr since withou head dim coalescing affected
	# thinking again block size M will acocount for coaelscing so need not worry 
	# wrong heads are issue for parallelization. better to head,seq format


	k=tl.load(k_block_ptr,boundary_check=(1,),padding_option="zero")*sm_scale
	v=tl.load(v_block_ptr,boundary_check=(0,),padding_option="zero")

	
	dk=tl.load(dk_block_ptr,boundary_check=(1,),padding_option="zero")
	dv=tl.load(dv_block_ptr,boundary_check=(0,),padding_option="zero")
	

	acc_dv=tl.zeros((BLOCK_N,HEAD_DIM),tl.float32)
	acc_dk=tl.zeros((BLOCK_N,HEAD_DIM),tl.float32)
	

	for _ in range(0,seq_len,BLOCK_M):
		#laod do dq and q for calculating
		dq=tl.load(dq_block_ptr,boundary_check=(1,),padding_option="zero")
		do=tl.load(do_block_ptr,boundary_check=(1,),padding_option="zero")
		q=tl.load(q_block_ptr,boundary_check=(1,),padding_option="zero")

		odo_sum=tl.load(odo_sum_block_ptr,boundary_check=(1,),padding_option="zero")
		lse=tl.load(lse_block_ptr,BLOCK_M)

		#calculate similarlity matrix sij=<qi,kj> shape is of B_MxB_N
		s=tl.dot(q,k)
		
		#calculate prob via softmax here lse is log(exp(max)+Z),Z is sum of logits stored in forward
		p=tl.math.exp(s-lse[:,None])
		
		#observe that oi=aijvj +k .This gives us dvj= sum_i aijdoi. A^Tdo
		tl.dot(tl.trans(p),do,acc_dv)

		# To calcualate dL/daij we have <doi,dvj> 
		dp=tl.dot(do,tl.trans(v))

		# dL/dsij where sij=<qi,kj> . Using chain rule and total derivative we have dL/dsij =sum k dL/daik.daik/dsij
		# This is sumk dpij.pij(detlta(j,k)-pik)=pij(dpij-D) where D= <1,dO.O>
		ds=p*(dp-odo_sum) #

		#acc grad w.r.t q into dk
		tl.dot(tl.trans(ds),q*sm_scale,acc_dk)

		#accumulate grad w.r.t k into dq
		dq=dq+tl.dot(ds,k).to(tl.bfloat16)

		tl.store(dq_block_ptr,dq.to(tl.bfloat16), boundary_check=(0,))
		
		#advance ptrs
		dq_block_ptr=tl.advance(dq_block_ptr,(BLOCK_M,0))
		q_block_ptr=tl.advance(q_block_ptr,(BLOCK_M,0))
		odo_block_ptr=tl.advance(odo_block_ptr,(BLOCK_M,))
		lse_block_ptr=tl.advance(lse_block_ptr,(BLOCK_M,))

	#store results
	tl.store(dk_block_ptr,dk.to(tl.bfloat16), boundary_check=(0,))
	tl.store(dv_block_ptr,dv.to(tl.bfloat16), boundary_check=(0,))

@triton.jit
def calc_odo_sum(o_ptr,do_ptr,odo_sum_ptr,seq_len,HEAD_DIM:tl.constexpr,BLOCK_K:tl.constexpr):
	pid=tl.program_id(axis=0)
	pid_h=tl.program_id(axis=2)
	pid_s=tl.program_id(axis=1)

	stride_tok=tl.num_programs(axis=2)*HEAD_DIM
	stride_h=tl.num_programs(axis=1)*seq_len

	o_ptr_b=o_ptr+pid_s*stride_tok+pid_h
	do_ptr_b=do_ptr+pid_s*stride_tok+pid_h*HEAD_DIM

	odo_sum_ptr_b=odo_sum_ptr+pid_h*stride_h + pid_s*seq_len

	o_block_ptr=tl.make_block_ptr(base=o_ptr_b,shape=(seq_len,HEAD_DIM),
							   strides=(stride_tok,0),offsets=(pid*BLOCK_K,0),
							   shape=(BLOCK_K,HEAD_DIM),order=(1,0))

	do_block_ptr=tl.make_block_ptr(base=do_ptr_b,shape=(seq_len,HEAD_DIM),
							   strides=(stride_tok,0),offsets=(pid*BLOCK_K,0),
							   shape=(BLOCK_K,HEAD_DIM),order=(1,0))	
	
	odo_sum_block_ptr=tl.make_block_ptr(base=odo_sum_ptr_b,shape=(seq_len,),
							   strides=(0,),offsets=(pid*BLOCK_K,),
							   boshape=(BLOCK_K,),order=(0,))
	
	o=tl.load(o_block_ptr)
	do=tl.load(do_block_ptr)

	odo=tl.sum(o*do,axis=1)

	tl.store(odo_sum_block_ptr,odo,boundary_check=(0,))


def flash_att_backward(q,k,v,o,do,lse_sum,num_seq,BLOCK_M=64,BLOCK_N=128,BLOCK_K=512):
	cum_seq_len,heads,head_dim=q.shape
	seq_len=cum_seq_len//num_seq

	dq,dk,dv,odo_sum=torch.empty_like(q),torch.empty_like(k),torch.empty_like(v),torch.empty_like(lse_sum)
	grid=lambda meta:(triton.cdiv(seq_len,BLOCK_K),num_docs,heads)

	calc_odo_sum[grid](o,do,odo_sum,seq_len,head_dim,BLOCK_K=BLOCK_K,num_warps=4,num_stages=4)
	sm_scale=1/(head_dim**0.5)
	
	grid=lambda meta:(triton.cdiv(seq_len,BLOCK_N),num_docs,heads)

	flash_att_backward_kernel[grid](q,k,v,dq,dk,dv,do,lse_sum,odo_sum,
								 seq_len,sm_scale,head_dim,BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N,
        num_warps=4, num_stages=2)

	return dq,dk,dv


SEQ_LEN=8
NUM_HEADS=12
HEAD_DIM=128
DTYPE=torch.bfloat16

q = torch.randn(1024*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
k = torch.randn(1024*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
v = torch.randn(1024*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
o = torch.randn(1024*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
do = torch.randn(1024*SEQ_LEN, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
lse_sum=torch.randn(NUM_HEADS,1024*SEQ_LEN,device=DEVICE, dtype=DTYPE)

flash_att_backward(q,k,v,o)


SyntaxError: keyword argument repeated: shape (52070514.py, line 66)

In [6]:
import torch
import triton
import triton.language as tl
from torch.nn.attention.flex_attention import flex_attention, create_block_mask

DEVICE = "cuda:2"
DTYPE = torch.bfloat16
HEAD_DIM = 128
NUM_HEADS = 12
NUM_DOCS = 8
SEQ_LEN = 1024

# -------------------------------------------------------------------------
# 1. CUSTOM TRITON KERNEL
# -------------------------------------------------------------------------
@triton.jit
def flash_att_intra_doc_masked(
    q_ptr, k_ptr, v_ptr, o_ptr, cuseq_ptr,
    sm_scale, HEAD_DIM: tl.constexpr, BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr
):
    pid = tl.program_id(axis=0)      
    pid_doc = tl.program_id(axis=1)  
    pid_h = tl.program_id(axis=2)    

    start = tl.load(cuseq_ptr + pid_doc)
    end = tl.load(cuseq_ptr + pid_doc + 1)
    doc_len = end - start

    stride_tok = tl.num_programs(axis=2) * HEAD_DIM 
    
    base_offset = start * stride_tok + pid_h * HEAD_DIM
    q_ptr_b = q_ptr + base_offset
    k_ptr_b = k_ptr + base_offset
    v_ptr_b = v_ptr + base_offset
    o_ptr_b = o_ptr + base_offset

    q_block_ptr = tl.make_block_ptr(
        base=q_ptr_b, shape=(doc_len, HEAD_DIM), strides=(stride_tok, 1),
        offsets=(pid * BLOCK_M, 0), block_shape=(BLOCK_M, HEAD_DIM), order=(1, 0)
    )
    k_block_ptr = tl.make_block_ptr(
        base=k_ptr_b, shape=(HEAD_DIM, doc_len), strides=(1, stride_tok),
        offsets=(0, 0), block_shape=(HEAD_DIM, BLOCK_N), order=(0, 1)
    )
    v_block_ptr = tl.make_block_ptr(
        base=v_ptr_b, shape=(doc_len, HEAD_DIM), strides=(stride_tok, 1),
        offsets=(0, 0), block_shape=(BLOCK_N, HEAD_DIM), order=(1, 0)
    )
    o_block_ptr = tl.make_block_ptr(
        base=o_ptr_b, shape=(doc_len, HEAD_DIM), strides=(stride_tok, 1),
        offsets=(pid * BLOCK_M, 0), block_shape=(BLOCK_M, HEAD_DIM), order=(1, 0)
    )

    qk_scale = sm_scale * 1.44269504
    q = tl.load(q_block_ptr, boundary_check=(0,), padding_option="zero")
    q = (q * qk_scale).to(tl.bfloat16)
    
    acc = tl.zeros((BLOCK_M, HEAD_DIM), tl.float32)
    max_val = tl.full((BLOCK_M,), float("-inf"), tl.float32)
    sum_val = tl.zeros((BLOCK_M,), tl.float32)

    q_offset = pid * BLOCK_M + tl.arange(0, BLOCK_M)
    doc_bias = tl.where(q_offset < doc_len, 0.0, -10000.0) 

    # 1. Safe Loop
    limit = pid * BLOCK_M
    for i in range(0, limit, BLOCK_N): 
        k = tl.load(k_block_ptr, boundary_check=(1,), padding_option="zero")
        v = tl.load(v_block_ptr, boundary_check=(0,), padding_option="zero")

        scores = tl.dot(q, k) 
        
        new_max = tl.maximum(max_val, tl.max(scores, axis=1)) 
        scores_exp = tl.math.exp2(scores - new_max[:, None])
        update_factor = tl.math.exp2(max_val - new_max)

        acc = acc * update_factor[:, None] + tl.dot(scores_exp.to(tl.bfloat16), v)
        sum_val = sum_val * update_factor + tl.sum(scores_exp, axis=1)
        max_val = new_max
        
        k_block_ptr = tl.advance(k_block_ptr, (0, BLOCK_N))
        v_block_ptr = tl.advance(v_block_ptr, (BLOCK_N, 0))

    # 2. Causal Loop
    limit_causal = tl.minimum((pid + 1) * BLOCK_M, doc_len)
    
    for i in range(limit, limit_causal, BLOCK_N): 
        k = tl.load(k_block_ptr, boundary_check=(1,), padding_option="zero")
        v = tl.load(v_block_ptr, boundary_check=(0,), padding_option="zero")

        k_offset = i + tl.arange(0, BLOCK_N)
        causal_bias = tl.where(q_offset[:, None] >= k_offset[None, :], 0.0, -10000.0) 
        
        scores = tl.dot(q, k) + causal_bias + doc_bias[:, None]       

        new_max = tl.maximum(max_val, tl.max(scores, axis=1))
        scores_exp = tl.math.exp2(scores - new_max[:, None])
        update_factor = tl.math.exp2(max_val - new_max)

        acc = acc * update_factor[:, None] + tl.dot(scores_exp.to(tl.bfloat16), v)
        sum_val = sum_val * update_factor + tl.sum(scores_exp, axis=1)
        max_val = new_max

        k_block_ptr = tl.advance(k_block_ptr, (0, BLOCK_N))
        v_block_ptr = tl.advance(v_block_ptr, (BLOCK_N, 0))

    acc = acc / sum_val[:, None]
    tl.store(o_block_ptr, acc.to(tl.bfloat16), boundary_check=(0,))

def triton_attention(q, k, v, cu_seq_lens):
    total_tokens, num_heads, head_dim = q.shape
    num_docs = cu_seq_lens.shape[0] - 1
    out = torch.empty_like(q)
    
    seq_lens = cu_seq_lens[1:] - cu_seq_lens[:-1]
    max_len = seq_lens.max().item()
    
    BLOCK_M, BLOCK_N = 64, 64
    grid = (triton.cdiv(max_len, BLOCK_M), num_docs, num_heads)
    sm_scale = 1.0 / (head_dim ** 0.5)
    
    flash_att_intra_doc_masked[grid](
        q, k, v, out, cu_seq_lens,
        sm_scale, HEAD_DIM=head_dim, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N,
        num_warps=4, num_stages=3
    )
    return out


# -------------------------------------------------------------------------
# 1. CUSTOM TRITON KERNEL (WITH AUTOTUNER)
# -------------------------------------------------------------------------
@triton.autotune(
    configs=[
        # The ultimate A6000 config (96KB SRAM)
        triton.Config({'BLOCK_M': 128, 'BLOCK_N': 64}, num_warps=8, num_stages=3),
        # Slightly lower register pressure
        triton.Config({'BLOCK_M': 128, 'BLOCK_N': 64}, num_warps=4, num_stages=3),
        # Deeper pipeline for memory latency
        triton.Config({'BLOCK_M': 64,  'BLOCK_N': 64}, num_warps=4, num_stages=3),
        triton.Config({'BLOCK_M': 64,  'BLOCK_N': 64}, num_warps=8, num_stages=3),
        # If SRAM allows slightly more breathing room
        triton.Config({'BLOCK_M': 128, 'BLOCK_N': 32}, num_warps=8, num_stages=3),
    ],
    key=['doc_len', 'HEAD_DIM'],
)
@triton.jit
def flash_att_intra_doc_masked(
    q_ptr, k_ptr, v_ptr, o_ptr, cuseq_ptr,
    sm_scale, HEAD_DIM: tl.constexpr, BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr
):
    pid = tl.program_id(axis=0)      
    pid_doc = tl.program_id(axis=1)  
    pid_h = tl.program_id(axis=2)    

    start = tl.load(cuseq_ptr + pid_doc)
    end = tl.load(cuseq_ptr + pid_doc + 1)
    doc_len = end - start

    stride_tok = tl.num_programs(axis=2) * HEAD_DIM 
    
    base_offset = start * stride_tok + pid_h * HEAD_DIM
    q_ptr_b = q_ptr + base_offset
    k_ptr_b = k_ptr + base_offset
    v_ptr_b = v_ptr + base_offset
    o_ptr_b = o_ptr + base_offset

    q_block_ptr = tl.make_block_ptr(
        base=q_ptr_b, shape=(doc_len, HEAD_DIM), strides=(stride_tok, 1),
        offsets=(pid * BLOCK_M, 0), block_shape=(BLOCK_M, HEAD_DIM), order=(1, 0)
    )
    k_block_ptr = tl.make_block_ptr(
        base=k_ptr_b, shape=(HEAD_DIM, doc_len), strides=(1, stride_tok),
        offsets=(0, 0), block_shape=(HEAD_DIM, BLOCK_N), order=(0, 1)
    )
    v_block_ptr = tl.make_block_ptr(
        base=v_ptr_b, shape=(doc_len, HEAD_DIM), strides=(stride_tok, 1),
        offsets=(0, 0), block_shape=(BLOCK_N, HEAD_DIM), order=(1, 0)
    )
    o_block_ptr = tl.make_block_ptr(
        base=o_ptr_b, shape=(doc_len, HEAD_DIM), strides=(stride_tok, 1),
        offsets=(pid * BLOCK_M, 0), block_shape=(BLOCK_M, HEAD_DIM), order=(1, 0)
    )

    qk_scale = sm_scale * 1.44269504
    q = tl.load(q_block_ptr, boundary_check=(0,), padding_option="zero")
    q = (q * qk_scale).to(tl.bfloat16)
    
    acc = tl.zeros((BLOCK_M, HEAD_DIM), tl.float32)
    max_val = tl.full((BLOCK_M,), float("-inf"), tl.float32)
    sum_val = tl.zeros((BLOCK_M,), tl.float32)

    q_offset = pid * BLOCK_M + tl.arange(0, BLOCK_M)
    doc_bias = tl.where(q_offset < doc_len, 0.0, -10000.0) 

    # 1. Safe Loop
    limit = pid * BLOCK_M
    for i in range(0, limit, BLOCK_N): 
        k = tl.load(k_block_ptr, boundary_check=(1,), padding_option="zero")
        v = tl.load(v_block_ptr, boundary_check=(0,), padding_option="zero")

        scores = tl.dot(q, k) + doc_bias[:, None] 
        
        new_max = tl.maximum(max_val, tl.max(scores, axis=1)) 
        scores_exp = tl.math.exp2(scores - new_max[:, None])
        update_factor = tl.math.exp2(max_val - new_max)

        acc = acc * update_factor[:, None] + tl.dot(scores_exp.to(tl.bfloat16), v)
        sum_val = sum_val * update_factor + tl.sum(scores_exp, axis=1)
        max_val = new_max
        
        k_block_ptr = tl.advance(k_block_ptr, (0, BLOCK_N))
        v_block_ptr = tl.advance(v_block_ptr, (BLOCK_N, 0))

    # 2. Causal Loop
    limit_causal = tl.minimum((pid + 1) * BLOCK_M, doc_len)
    
    for i in range(limit, limit_causal, BLOCK_N): 
        k = tl.load(k_block_ptr, boundary_check=(1,), padding_option="zero")
        v = tl.load(v_block_ptr, boundary_check=(0,), padding_option="zero")

        k_offset = i + tl.arange(0, BLOCK_N)
        causal_bias = tl.where(q_offset[:, None] >= k_offset[None, :], 0.0, -10000.0) 
        
        scores = tl.dot(q, k) + causal_bias + doc_bias[:, None]       

        new_max = tl.maximum(max_val, tl.max(scores, axis=1))
        scores_exp = tl.math.exp2(scores - new_max[:, None])
        update_factor = tl.math.exp2(max_val - new_max)

        acc = acc * update_factor[:, None] + tl.dot(scores_exp.to(tl.bfloat16), v)
        sum_val = sum_val * update_factor + tl.sum(scores_exp, axis=1)
        max_val = new_max

        k_block_ptr = tl.advance(k_block_ptr, (0, BLOCK_N))
        v_block_ptr = tl.advance(v_block_ptr, (BLOCK_N, 0))

    acc = acc / sum_val[:, None]
    tl.store(o_block_ptr, acc.to(tl.bfloat16), boundary_check=(0,))


def triton_attention(q, k, v, cu_seq_lens):
    total_tokens, num_heads, head_dim = q.shape
    num_docs = cu_seq_lens.shape[0] - 1
    out = torch.empty_like(q)
    
    seq_lens = cu_seq_lens[1:] - cu_seq_lens[:-1]
    max_len = seq_lens.max().item()
    
    # Dynamic grid matching the autotuner's chosen BLOCK_M
    grid = lambda META: (triton.cdiv(max_len, META['BLOCK_M']), num_docs, num_heads)
    sm_scale = 1.0 / (head_dim ** 0.5)
    
    flash_att_intra_doc_masked[grid](
        q, k, v, out, cu_seq_lens,
        sm_scale, HEAD_DIM=head_dim
    )
    return out
# -------------------------------------------------------------------------
# 2. COMPILED FLEX ATTENTION SETUP
# -------------------------------------------------------------------------
# Explicitly apply torch.compile to flex_attention itself, not the wrapper
compiled_flex = torch.compile(flex_attention, dynamic=False)

def setup_flex_attention(q, k, v, doc_ids):
    q_f = q.unsqueeze(0).transpose(1, 2) 
    k_f = k.unsqueeze(0).transpose(1, 2)
    v_f = v.unsqueeze(0).transpose(1, 2)
    
    def doc_causal_mask(b, h, q_idx, kv_idx):
        return (q_idx >= kv_idx) & (doc_ids[q_idx] == doc_ids[kv_idx])

    total_seq = q_f.shape[2]
    block_mask = create_block_mask(doc_causal_mask, B=1, H=1, Q_LEN=total_seq, KV_LEN=total_seq, device=DEVICE)

    def run_flex():
        return compiled_flex(q_f, k_f, v_f, block_mask=block_mask)
        
    # FORCE multiple warmup passes to ensure Dynamo finishes tracing
    print("Warming up FlexAttention (Tracing + Compilation)...")
    for _ in range(3):
        _ = run_flex()
    print("Warmup Complete.")
    
    return run_flex

# -------------------------------------------------------------------------
# 3. BENCHMARK HARNESS
# -------------------------------------------------------------------------
total_tokens = SEQ_LEN * NUM_DOCS

lengths = torch.full((NUM_DOCS,), SEQ_LEN, dtype=torch.int32, device=DEVICE)
cu_seq_lens = torch.zeros(NUM_DOCS + 1, dtype=torch.int32, device=DEVICE)
cu_seq_lens[1:] = torch.cumsum(lengths, dim=0)

doc_ids = torch.zeros(total_tokens, dtype=torch.int32, device=DEVICE)
for i in range(NUM_DOCS):
    doc_ids[cu_seq_lens[i]:cu_seq_lens[i+1]] = i
    
q = torch.randn(total_tokens, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
k = torch.randn(total_tokens, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
v = torch.randn(total_tokens, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)

# Setup
_ = triton_attention(q, k, v, cu_seq_lens)
run_flex = setup_flex_attention(q, k, v, doc_ids)

# Correctness verification (Using standard 2e-2 tolerance for BF16)
out_triton = triton_attention(q, k, v, cu_seq_lens)
out_flex_raw = run_flex()
out_flex = out_flex_raw.transpose(1, 2).squeeze(0)

max_err = (out_triton - out_flex).abs().max().item()
ok = torch.allclose(out_triton, out_flex, atol=2e-2, rtol=2e-2)

# Benchmark Iterations
ms_triton = triton.testing.do_bench(lambda: triton_attention(q, k, v, cu_seq_lens))
ms_flex = triton.testing.do_bench(lambda: run_flex())

# Causal FLOPs for multiple stacked documents
flops = NUM_DOCS * (2 * NUM_HEADS * (SEQ_LEN ** 2) * HEAD_DIM)
tflops_triton = flops / (ms_triton * 1e-3) / 1e12
tflops_flex = flops / (ms_flex * 1e-3) / 1e12

print(f"\n{'Seq':>6} | {'Triton(ms)':>10} {'Flex(ms)':>9} | {'Tri TFLOPs':>10} {'Flex TFLOPs':>11} | {'Ratio':>6} {'MaxErr':>8} {'OK':>4}")
print("-" * 80)
print(f"{SEQ_LEN:>6} | {ms_triton:>10.4f} {ms_flex:>9.4f} | {tflops_triton:>10.1f} {tflops_flex:>11.1f} | {ms_triton/ms_flex:>6.2f} {max_err:>8.4f} {str(ok):>4}")

Warming up FlexAttention (Tracing + Compilation)...
Warmup Complete.

   Seq | Triton(ms)  Flex(ms) | Tri TFLOPs Flex TFLOPs |  Ratio   MaxErr   OK
--------------------------------------------------------------------------------
  1024 |     0.3911    0.3347 |       65.9        77.0 |   1.17   0.0156 True


In [ ]:
best = flash_att_intra_doc_masked.best_config

print("Block Sizes:", best.kwargs)
print("Num Warps:  ", best.num_warps)
print("Num Stages: ", best.num_stages)

Block Sizes: {'BLOCK_M': 128, 'BLOCK_N': 32}
Num Warps:   8
Num Stages:  3


In [ ]:
import torch
import torch.nn.functional as F
import triton
import triton.language as tl
from torch.nn.attention import sdpa_kernel, SDPBackend

# 1. Re-enable the Autotuner with aggressive configs for A100/H100
@triton.autotune(
    configs=[
        # # A6000 / Ampere Optimized (Requires < 100KB SRAM)
        # triton.Config({'BLOCK_SIZE_Q': 128, 'BLOCK_SIZE_KV': 64}, num_warps=8, num_stages=2),
        # triton.Config({'BLOCK_SIZE_Q': 64, 'BLOCK_SIZE_KV': 128}, num_warps=8, num_stages=2),
        # triton.Config({'BLOCK_SIZE_Q': 64, 'BLOCK_SIZE_KV': 64},  num_warps=4, num_stages=3),
        # triton.Config({'BLOCK_SIZE_Q': 64, 'BLOCK_SIZE_KV': 64},  num_warps=8, num_stages=2),
        triton.Config({'BLOCK_SIZE_Q': 128, 'BLOCK_SIZE_KV': 64},  num_warps=8, num_stages=2),

    ],
    key=['qn', 'kvn', 'HEAD_DIM'],
)
@triton.jit
def flash_att_kernel_batched(
    q_ptr, k_ptr, v_ptr, out_ptr, 
    qn, kvn, sm_scale, 
    NUM_HEADS: tl.constexpr, HEAD_DIM: tl.constexpr, 
    BLOCK_SIZE_Q: tl.constexpr, BLOCK_SIZE_KV: tl.constexpr
):
    pid   = tl.program_id(axis=0)  # Was axis 2
    pid_h = tl.program_id(axis=1)  # Stays axis 1
    pid_b = tl.program_id(axis=2)  # Was axis 0

    # Offset pointers
    q_ptr_b = q_ptr + pid_b * NUM_HEADS * qn * HEAD_DIM + pid_h * qn * HEAD_DIM  
    k_ptr_b = k_ptr + pid_b * NUM_HEADS * kvn * HEAD_DIM + pid_h * HEAD_DIM * kvn  
    v_ptr_b = v_ptr + pid_b * NUM_HEADS * kvn * HEAD_DIM + pid_h * HEAD_DIM * kvn 
    out_ptr_b = out_ptr + pid_b * NUM_HEADS * qn * HEAD_DIM + pid_h * HEAD_DIM * qn 

    q_block_ptr = tl.make_block_ptr(
        base=q_ptr_b, shape=(qn, HEAD_DIM), strides=(HEAD_DIM, 1),
        offsets=(pid * BLOCK_SIZE_Q, 0), block_shape=(BLOCK_SIZE_Q, HEAD_DIM), order=(1, 0)
    )
    
    k_block_ptr = tl.make_block_ptr(
        base=k_ptr_b,
        shape=(HEAD_DIM, kvn),           # Swapped shape
        strides=(1, HEAD_DIM),           # Swapped strides (row_stride=1, col_stride=HEAD_DIM)
        offsets=(0, 0),
        block_shape=(HEAD_DIM, BLOCK_SIZE_KV), # Swapped block shape
        order=(0, 1)                     # THE TRICK: Tell Triton dim 0 (HEAD_DIM) is contiguous!
    )
    v_block_ptr = tl.make_block_ptr(
        base=v_ptr_b, shape=(kvn, HEAD_DIM), strides=(HEAD_DIM, 1),
        offsets=(0, 0), block_shape=(BLOCK_SIZE_KV, HEAD_DIM), order=(1, 0)
    )
    
    # 2. Pre-scale Q outside the loop to save compute
    qk_scale = sm_scale * 1.44269504
    q = tl.load(q_block_ptr)
    q = (q * qk_scale).to(tl.bfloat16)
    
    sum_val = tl.zeros((BLOCK_SIZE_Q,), tl.float32)
    acc = tl.zeros((BLOCK_SIZE_Q, HEAD_DIM), tl.float32)
    max_val = tl.full((BLOCK_SIZE_Q,), float("-inf"), tl.float32)

    for i in range(0, kvn, BLOCK_SIZE_KV):
        k = tl.load(k_block_ptr)
        v = tl.load(v_block_ptr)

        # Removed *sm_scale from here
        scores = tl.dot(q, k) 
        new_max = tl.maximum(max_val, tl.max(scores, axis=1))

        scores_exp = tl.math.exp2(scores - new_max[:, None])
        update_factor = tl.math.exp2(max_val - new_max)

        acc = acc * update_factor[:, None] 
        acc=tl.dot(scores_exp.to(tl.bfloat16), v,acc)
        sum_val = sum_val * update_factor + tl.sum(scores_exp, axis=1)

        max_val = new_max
            
        k_block_ptr = tl.advance(k_block_ptr, (0, BLOCK_SIZE_KV))        
        v_block_ptr = tl.advance(v_block_ptr, (BLOCK_SIZE_KV, 0))

    acc = acc / sum_val[:, None]

    out_block_ptr = tl.make_block_ptr(
        base=out_ptr_b, shape=(qn, HEAD_DIM), strides=(HEAD_DIM, 1),
        offsets=(pid * BLOCK_SIZE_Q, 0), block_shape=(BLOCK_SIZE_Q, HEAD_DIM), order=(1, 0)
    )
    
    tl.store(out_block_ptr, acc.to(tl.bfloat16))

def flash_att_block(q, k, v):
    batch_size, num_head, num_q, head_dim = q.shape
    num_kv = k.shape[-2]
    sm_scale = 1.0 / (head_dim ** 0.5)
    out = torch.empty_like(q)
    
    # 3. Dynamic grid mapping using META to adapt to the autotuner's choices
    grid = lambda META: (
            triton.cdiv(num_q, META['BLOCK_SIZE_Q']), 
            num_head, 
            batch_size
        )    
    flash_att_kernel_batched[grid](
        q, k, v, out, 
        num_q, num_kv, sm_scale,
        NUM_HEADS=num_head, HEAD_DIM=head_dim
    )
    return out

q,k,v=torch.randn(2,4,1024,64,device="cuda:2").to(torch.bfloat16).contiguous(),torch.randn(2,4,1024,64,device="cuda:2").to(torch.bfloat16).contiguous(),torch.randn(2,4,1024,64,device="cuda:2").to(torch.bfloat16).contiguous()
ms_triton = triton.testing.do_bench(lambda: flash_att_block(q,k,v))

ms_triton


import torch
import torch.nn.functional as F
import triton
from torch.nn.attention import sdpa_kernel, SDPBackend

DEVICE = "cuda:2"
DTYPE = torch.bfloat16
HEAD_DIM = 128
BATCH_SIZE=12
NUM_HEAD=12

def ref_attention(q, k, v):
    """Single-head reference, 2D in/out, via SDPA's FlashAttention backend."""
    with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
        return F.scaled_dot_product_attention(
         q,k,v  
        )

def bench_one(seq_len):
    q, k, v = (torch.randn(BATCH_SIZE,NUM_HEAD,seq_len, HEAD_DIM, device=DEVICE, dtype=DTYPE) for _ in range(3))

    # --- correctness ---
    out_triton = flash_att_block(q, k, v).float()
    out_ref    = ref_attention(q, k, v).float()
    max_err = (out_triton - out_ref).abs().max().item()
    ok = torch.allclose(out_triton, out_ref, atol=2e-2, rtol=2e-2)

    # --- timing ---
    ms_triton = triton.testing.do_bench(lambda: flash_att_block(q, k, v))
    ms_flash  = triton.testing.do_bench(lambda: ref_attention(q, k, v))

    # --- TFLOP/s (attention FLOPs ≈ 4 * seq^2 * head_dim: two matmuls) ---
    flops = 4 * BATCH_SIZE*NUM_HEAD* seq_len * seq_len * HEAD_DIM
    tflops_triton = flops / (ms_triton * 1e-3) / 1e12
    tflops_flash  = flops / (ms_flash  * 1e-3) / 1e12

    return seq_len, ms_triton, ms_flash, tflops_triton, tflops_flash, max_err, ok

print(f"{'seq':>6} {'triton(ms)':>11} {'flash(ms)':>10} {'tri TFLOPs':>11} "
      f"{'fl TFLOPs':>10} {'ratio':>6} {'maxerr':>8} {'ok':>4}")
for seq in [64,128,256,512,1024,2048]:
    s, mt, mf, tt, tf, err, ok = bench_one(seq)
    print(f"{s:>6} {mt:>11.4f} {mf:>10.4f} {tt:>11.1f} {tf:>10.1f} "
          f"{mt/mf:>6.2f} {err:>8.4f} {str(ok):>4}")


   seq  triton(ms)  flash(ms)  tri TFLOPs  fl TFLOPs  ratio   maxerr   ok
    64      0.0248     0.0231        12.2       13.1   1.07   1.9160 False
   128      0.0355     0.0330        34.1       36.6   1.08   0.0078 True
   256      0.0732     0.0714        66.0       67.7   1.03   0.0039 True
   512      0.2018     0.1877        95.8      103.0   1.07   0.0039 True
  1024      0.7625     0.7018       101.4      110.2   1.09   0.0039 True
  2048      3.0162     2.7470       102.5      112.6   1.10   0.0020 True


In [ ]:
best = flash_att_kernel_batched.best_config

print("Block Sizes:", best.kwargs)
print("Num Warps:  ", best.num_warps)
print("Num Stages: ", best.num_stages)

Block Sizes: {'BLOCK_SIZE_Q': 128, 'BLOCK_SIZE_KV': 64}
Num Warps:   8
Num Stages:  3


In [ ]:
@triton.jit
def softmax_kernel(x_ptr,out_ptr,col_size,BLOCK_SIZE:tl.constexpr):
  pid=tl.program_id(axis=0)

  block_idx=pid*col_size
  offset=block_idx+tl.arange(0,BLOCK_SIZE)

  mask=offset<block_idx+col_size

  x=tl.load(x_ptr+offset,mask,other=float("-inf"))

  max=tl.max(x,axis=0)
  x_exp=tl.exp(x-max)
  normalizer=tl.sum(x_exp,axis=0)
  softmax=x_exp/normalizer

  tl.store(out_ptr+offset,softmax,mask)

  return


def softmax(x):
  out=torch.empty_like(x)
  n_rows,n_cols=x.shape
  # grid = lambda meta: (triton.cdiv(n_rows, meta['BLOCK_SIZE']),)

  softmax_kernel[(n_rows,)](x,out,n_cols,BLOCK_SIZE=triton.next_power_of_2(n_cols))
  return out


x=torch.randn(int(10),20).to(DEVICE)
out=softmax(x)
out_torch=F.softmax(x,dim=-1)
out[0],out_torch[0],out[0]==out_torch[0]
torch.allclose(out,out_torch,1e-5)

True

In [ ]:
#myn,nxk->mxk
#TXT,TXT->TXT T Block Size

@triton.autotune(
configs=[
  triton.Config({'ROW_BLOCK_SIZE':64,'COL_BLOCK_SIZE':64,'K_BLOCK_SIZE':64},  num_warps=4, num_stages=1),
  triton.Config({'ROW_BLOCK_SIZE':64,'COL_BLOCK_SIZE':64,'K_BLOCK_SIZE':128}, num_warps=4, num_stages=2),
  triton.Config({'ROW_BLOCK_SIZE':128,'COL_BLOCK_SIZE':64,'K_BLOCK_SIZE':64}, num_warps=4, num_stages=2),
],
  key=['xm','xn','yk'],
)

@triton.jit
def matmul_kernel(x_ptr,y_ptr,out_ptr,xm,xn,yn,yk,
           ROW_BLOCK_SIZE:tl.constexpr,COL_BLOCK_SIZE:tl.constexpr,K_BLOCK_SIZE:tl.constexpr):

  pid_m=tl.program_id(axis=0)
  pid_n=tl.program_id(axis=1)

  r_idx=pid_m*ROW_BLOCK_SIZE
  c_idx=pid_n*COL_BLOCK_SIZE
  acc=tl.zeros((ROW_BLOCK_SIZE,COL_BLOCK_SIZE),tl.float32)

  for j in range(0,xn,K_BLOCK_SIZE):
        x_block_ptr=tl.make_block_ptr(base=x_ptr,
                                      shape=(xm,xn),
                                      strides=(xn,1),
                                      offsets=(r_idx,j),
                                      block_shape=(ROW_BLOCK_SIZE,K_BLOCK_SIZE),
                                      order=(1,0))

        x_tile=tl.load(x_block_ptr,boundary_check=(0,1),padding_option="zero")

        y_block_ptr=tl.make_block_ptr(base=y_ptr,
                                      shape=(yn,yk),
                                      strides=(yk,1),
                                      offsets=(j,c_idx),
                                      block_shape=(K_BLOCK_SIZE,COL_BLOCK_SIZE),
                                      order=(1,0))

        y_tile=tl.load(y_block_ptr,boundary_check=(0,1),padding_option="zero")

        acc=tl.dot(x_tile,y_tile,acc)


  out_block_ptr=tl.make_block_ptr(base=out_ptr,
                                  shape=(xm,yk),
                                  strides=(yk,1),
                                  offsets=(r_idx,c_idx),
                                  block_shape=(ROW_BLOCK_SIZE,COL_BLOCK_SIZE),
                                  order=(1,0))
  tl.store(out_block_ptr,acc.to(tl.float16),boundary_check=(0,1),)


def matmul(x,y):
  xm,xn=x.shape
  yn,yk=y.shape
  out=torch.empty((xm,yk),dtype=torch.float16,device="cuda:2")
  grid_2d = lambda meta: (triton.cdiv(xm, meta['ROW_BLOCK_SIZE']),triton.cdiv(yk, meta['COL_BLOCK_SIZE']))

  matmul_kernel[grid_2d](x,y,out,xm,xn,yn,yk)
  return out


x,y=torch.randn(32,16).to("cuda:2").to(dtype=torch.float16),torch.randn(16,4).to("cuda").to(dtype=torch.float16)
trit_mm=matmul(x,y)
torch_mm=torch.mm(x,y)
torch.allclose(trit_mm,torch_mm)


ValueError: Pointer argument (at 0) cannot be accessed from Triton (cpu tensor?)

In [ ]:
print(matmul_kernel.best_config)  # the winner

AttributeError: 'Autotuner' object has no attribute 'best_config'

In [ ]:
def gbps(ms,x,y,block_size):
  xm,xn=x.shape
  yn,yk=y.shape
  bytes_moved=(2*xm*xn*yk*x.element_size())/block_size
  return (bytes_moved/(ms*1e-3))/1e9


n_rows, n_cols,n_out_cols = 1823, 781 , 1823
x = torch.randn(n_rows, n_cols, device='cuda:2').half()
y = torch.randn(n_cols,n_out_cols,device="cuda:2").half()

f=torch.compile(lambda x,y:torch.mm(x,y))
f(x,y)
# for _ in range(5):
#   ms_triton = triton.testing.do_bench(lambda: matmul(x,y))

ms_triton = triton.testing.do_bench(lambda: matmul(x,y))
ms_torch  = triton.testing.do_bench(lambda: torch.mm(x,y))
ms_torch_compile=triton.testing.do_bench(lambda: f(x,y))

print(torch.cuda.get_device_name())
print(f"triton: {ms_triton:.4f} ms | {gbps(ms_triton, x,y,64):.0f} GB/s")
print(f"torch : {ms_torch:.4f} ms | {gbps(ms_torch, x,y,64):.0f} GB/s")
print(f"torch_compile: {ms_torch_compile:.4f} ms  | {gbps(ms_torch_compile,x,y,64):.0f} GB/S")

ValueError: Pointer argument (at 0) cannot be accessed from Triton (cpu tensor?)

In [ ]:
print(matmul_kernel.best_config)

ROW_BLOCK_SIZE: 64, COL_BLOCK_SIZE: 64, K_BLOCK_SIZE: 64, num_warps: 4, num_ctas: 1, num_stages: 1, maxnreg: None


In [ ]:
#NO batch one query in one block
@triton.jit
def flash_att_kernel(q_ptr,k_ptr,v_ptr,out_ptr,head_dim,num_q,num_kv,sm_scale,BLOCK_SIZE:tl.constexpr):

  pid=tl.program_id(axis=0)

  block_idx=pid
  block_ptr=tl.make_block_ptr(base=q_ptr,
                              shape=(num_q,head_dim),
                              strides=(head_dim,1),
                              offsets=(block_idx,0),
                              block_shape=(1,BLOCK_SIZE),
                              order=(1,0))
  q=tl.load(block_ptr)

  acc_out=tl.zeros((1,BLOCK_SIZE,),tl.float32)

  sum_val=float(0)
  maxm=-float("inf")


  k_block_ptr=tl.make_block_ptr(base=k_ptr,
                              shape=(num_kv,head_dim),
                              strides=(head_dim,1),
                              offsets=(0,0),
                              block_shape=(1,BLOCK_SIZE),
                              order=(1,0))

  v_block_ptr=tl.make_block_ptr(base=v_ptr,
                              shape=(num_kv,head_dim),
                              strides=(head_dim,1),
                              offsets=(0,0),
                              block_shape=(1,BLOCK_SIZE),
                              order=(1,0))

  for i in range(0,num_kv,1):
      #load k,v
      k=tl.load(k_block_ptr)
      v=tl.load(v_block_ptr)

#     calculate <q,k>
      score=tl.dot(q,tl.trans(k))
      score = tl.sum(score)*sm_scale

      #calc new max
      new_max=tl.maximum(maxm,score)

      #calc e^(ai-new_max) and e^(old_max-new_max)

      score_exp=tl.exp(score-new_max)
      new_exp=tl.exp(maxm-new_max)

      #Update score to sumi e^(xi-old_max)*e^(old_max-new_max)= sumi e^(xi-new_max)
      acc_out=new_exp*acc_out

      #Add new aivi given as xi+e^(ai-max)*vi
      acc_out+=score_exp*v

      maxm=new_max

      #Update sum si*e(old_max-new_max)+e^(ai-new_max)
      sum_val=new_exp*sum_val + score_exp

      v_block_ptr=tl.advance(v_block_ptr,(1,0))
      k_block_ptr=tl.advance(k_block_ptr,(1,0))

  acc_out=acc_out/sum_val

  out_block_ptr=tl.make_block_ptr(base=out_ptr,
                                  shape=(num_q,head_dim),
                                  strides=(head_dim,1),
                                  offsets=(block_idx,0),
                                  block_shape=(1,BLOCK_SIZE),
                                  order=(1,0))

  tl.store(out_block_ptr,acc_out)

def flash_att(q,k,v):
  num_q,head_dim=q.shape
  num_kv=k.shape[0]
  sm_scale = 1.0 / (head_dim ** 0.5)

  out=torch.empty_like(q)
  flash_att_kernel[(num_q,)](q,k,v,out,head_dim,num_q,num_kv,sm_scale,BLOCK_SIZE=head_dim)
  torch.cuda.synchronize()

  return out



q,k,v=torch.randn(2,32,device="cuda:2"),torch.randn(2,32,device="cuda:2"),torch.randn(2,32,device="cuda:2")

att_out=flash_att(q,k,v)

# print(q[0],k[0])
print(att_out)

output = F.scaled_dot_product_attention(q, k, v)
print(torch.allclose(att_out,output))
output


AcceleratorError: CUDA error: an illegal memory access was encountered
Search for `cudaErrorIllegalAddress' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
@triton.jit
def flash_block_att_kernel(q_ptr,k_ptr,v_ptr,out_ptr,qn,kvn,sm_scale,
                           HEAD_DIM:tl.constexpr,BLOCK_SIZE_Q:tl.constexpr,BLOCK_SIZE_KV:tl.constexpr):
    pid=tl.program_id(axis=0)

    q_block_ptr=tl.make_block_ptr(base=q_ptr,
                                  shape=(qn,HEAD_DIM),
                                  strides=(HEAD_DIM,1),
                                  offsets=(pid*BLOCK_SIZE_Q,0),
                                  block_shape=(BLOCK_SIZE_Q,HEAD_DIM),
                                 order=(1,0))
    k_block_ptr=tl.make_block_ptr(base=k_ptr,
                                  shape=(kvn,HEAD_DIM),
                                  strides=(HEAD_DIM,1),
                                  offsets=(0,0),
                                  block_shape=(BLOCK_SIZE_KV,HEAD_DIM),
                                 order=(1,0))
    v_block_ptr=tl.make_block_ptr(base=v_ptr,
                                  shape=(kvn,HEAD_DIM),
                                  strides=(HEAD_DIM,1),
                                  offsets=(0,0),
                                  block_shape=(BLOCK_SIZE_KV,HEAD_DIM),
                                 order=(1,0))
    
    q=tl.load(q_block_ptr)
    sum_val=tl.zeros((BLOCK_SIZE_Q,),tl.float32)
    acc=tl.zeros((BLOCK_SIZE_Q,HEAD_DIM),tl.float32)
    max_val=tl.full((BLOCK_SIZE_Q,) ,float("-inf"),tl.float32)

    for i in range(0,kvn,BLOCK_SIZE_KV):
        k,v=tl.load(k_block_ptr),tl.load(v_block_ptr) #M,H N,H

        scores=tl.dot(q,tl.trans(k))*sm_scale  #M,N
        new_max=tl.maximum(max_val,tl.max(scores,axis=1))  # M

        scores_exp=tl.exp(scores-new_max[:,None]) #M,N
        update_factor=tl.exp(max_val-new_max) #M

        acc=acc*update_factor[:,None] + tl.dot(scores_exp.to(tl.bfloat16), v) #M,H

        sum_val=sum_val*update_factor + tl.sum(scores_exp,axis=1) #M

        max_val=new_max
            
        k_block_ptr=tl.advance(k_block_ptr,(BLOCK_SIZE_KV,0))
        v_block_ptr=tl.advance(v_block_ptr,(BLOCK_SIZE_KV,0))

    acc=acc/sum_val[:,None]

    out_block_ptr=tl.make_block_ptr(base=out_ptr,
                                    shape=(qn,HEAD_DIM),
                                    strides=(HEAD_DIM,1),
                                    offsets=(pid*BLOCK_SIZE_Q,0),
                                    block_shape=(BLOCK_SIZE_Q,HEAD_DIM),
                                   order=(1,0))
    tl.store(out_block_ptr,acc.to(tl.bfloat16))



def flash_att_block(q, k, v, BLOCK_M=64, BLOCK_N=64):
    num_q, head_dim = q.shape
    num_kv = k.shape[0]
    sm_scale = 1.0 / (head_dim ** 0.5)
    out = torch.empty_like(q)
    grid = (triton.cdiv(num_q, BLOCK_M),)
    flash_block_att_kernel[grid](q, k, v, out, num_q, num_kv, sm_scale,
                           HEAD_DIM=head_dim, BLOCK_SIZE_Q=BLOCK_M, BLOCK_SIZE_KV=BLOCK_N)
    return out

# q,k,v=torch.randn(1024,64,device="cuda:2"),torch.randn(1024,64,device="cuda:2"),torch.randn(1024,64,device="cuda:2")

q,k,v=torch.randn(1024,64,device="cuda:2").to(torch.bfloat16),torch.randn(1024,64,device="cuda:2").to(torch.bfloat16),torch.randn(1024,64,device="cuda:2").to(torch.bfloat16)
ms_triton = triton.testing.do_bench(lambda: flash_att_block(q,k,v))

ms_triton



0.021503999829292297

In [ ]:
import torch
import torch.nn.functional as F
import triton
from torch.nn.attention import sdpa_kernel, SDPBackend

DEVICE = "cuda:2"
DTYPE = torch.bfloat16
HEAD_DIM = 64

def ref_attention(q, k, v):
    """Single-head reference, 2D in/out, via SDPA's FlashAttention backend."""
    with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
        return F.scaled_dot_product_attention(
           q,k,v
        )

def bench_one(seq_len):
    q, k, v = (torch.randn(seq_len, HEAD_DIM, device=DEVICE, dtype=DTYPE) for _ in range(3))

    # --- correctness ---
    out_triton = flash_att_block(q, k, v).float()
    out_ref    = ref_attention(q, k, v).float()
    max_err = (out_triton - out_ref).abs().max().item()
    ok = torch.allclose(out_triton, out_ref, atol=2e-2, rtol=2e-2)

    # --- timing ---
    ms_triton = triton.testing.do_bench(lambda: flash_att_block(q, k, v))
    ms_flash  = triton.testing.do_bench(lambda: ref_attention(q, k, v))

    # --- TFLOP/s (attention FLOPs ≈ 4 * seq^2 * head_dim: two matmuls) ---
    flops = 4 * seq_len * seq_len * HEAD_DIM
    tflops_triton = flops / (ms_triton * 1e-3) / 1e12
    tflops_flash  = flops / (ms_flash  * 1e-3) / 1e12

    return seq_len, ms_triton, ms_flash, tflops_triton, tflops_flash, max_err, ok

print(f"{'seq':>6} {'triton(ms)':>11} {'flash(ms)':>10} {'tri TFLOPs':>11} "
      f"{'fl TFLOPs':>10} {'ratio':>6} {'maxerr':>8} {'ok':>4}")
for seq in [512, 1024, 2048, 4096, 8192, 16384]:
    s, mt, mf, tt, tf, err, ok = bench_one(seq)
    print(f"{s:>6} {mt:>11.4f} {mf:>10.4f} {tt:>11.1f} {tf:>10.1f} "
          f"{mt/mf:>6.2f} {err:>8.4f} {str(ok):>4}")

   seq  triton(ms)  flash(ms)  tri TFLOPs  fl TFLOPs  ratio   maxerr   ok


ValueError: not enough values to unpack (expected 4, got 2)

In [ ]:



@triton.jit
def flash_att_kernel_batched(q_ptr,k_ptr,v_ptr,out_ptr,qn,kvn,sm_scale,NUM_HEADS:tl.constexpr,
                                HEAD_DIM:tl.constexpr,BLOCK_SIZE_Q:tl.constexpr,BLOCK_SIZE_KV:tl.constexpr):
    
    pid_b=tl.program_id(axis=0)
    pid_h=tl.program_id(axis=1)
    pid=tl.program_id(axis=2)

    q_ptr_b=q_ptr + pid_b*NUM_HEADS*qn*HEAD_DIM + pid_h*qn*HEAD_DIM  
    k_ptr_b=k_ptr + pid_b*NUM_HEADS*kvn*HEAD_DIM + pid_h*HEAD_DIM*kvn  
    v_ptr_b=v_ptr +  pid_b*NUM_HEADS*kvn*HEAD_DIM + pid_h*HEAD_DIM*kvn 

    out_ptr_b=out_ptr + pid_b*NUM_HEADS*qn*HEAD_DIM + pid_h*HEAD_DIM*qn 



    q_block_ptr=tl.make_block_ptr(base=q_ptr_b,
                                  shape=(qn,HEAD_DIM),
                                  strides=(HEAD_DIM,1),
                                  offsets=(pid*BLOCK_SIZE_Q,0),
                                  block_shape=(BLOCK_SIZE_Q,HEAD_DIM),
                                 order=(1,0))
    k_block_ptr=tl.make_block_ptr(base=k_ptr_b,
                                  shape=(kvn,HEAD_DIM),
                                  strides=(HEAD_DIM,1),
                                  offsets=(0,0),
                                  block_shape=(BLOCK_SIZE_KV,HEAD_DIM),
                                 order=(1,0))
    v_block_ptr=tl.make_block_ptr(base=v_ptr_b,
                                  shape=(kvn,HEAD_DIM),
                                  strides=(HEAD_DIM,1),
                                  offsets=(0,0),
                                  block_shape=(BLOCK_SIZE_KV,HEAD_DIM),
                                 order=(1,0))
    
    q=tl.load(q_block_ptr)
    sum_val=tl.zeros((BLOCK_SIZE_Q,),tl.float32)
    acc=tl.zeros((BLOCK_SIZE_Q,HEAD_DIM),tl.float32)
    max_val=tl.full((BLOCK_SIZE_Q,) ,float("-inf"),tl.float32)

    for i in range(0,kvn,BLOCK_SIZE_KV):
        k,v=tl.load(k_block_ptr),tl.load(v_block_ptr) #M,H N,H

        scores=tl.dot(q,tl.trans(k))*sm_scale  #M,N
        new_max=tl.maximum(max_val,tl.max(scores,axis=1))  # M

        scores_exp=tl.exp(scores-new_max[:,None]) #M,N
        update_factor=tl.exp(max_val-new_max) #M

        acc=acc*update_factor[:,None] + tl.dot(scores_exp.to(tl.bfloat16), v) #M,H

        sum_val=sum_val*update_factor + tl.sum(scores_exp,axis=1) #M

        max_val=new_max
            
        k_block_ptr=tl.advance(k_block_ptr,(BLOCK_SIZE_KV,0))
        v_block_ptr=tl.advance(v_block_ptr,(BLOCK_SIZE_KV,0))

    acc=acc/sum_val[:,None]

    out_block_ptr=tl.make_block_ptr(base=out_ptr_b,
                                    shape=(qn,HEAD_DIM),
                                    strides=(HEAD_DIM,1),
                                    offsets=(pid*BLOCK_SIZE_Q,0),
                                    block_shape=(BLOCK_SIZE_Q,HEAD_DIM),
                                   order=(1,0))
    
    tl.store(out_block_ptr,acc.to(tl.bfloat16))



def flash_att_block(q, k, v, BLOCK_M=64, BLOCK_N=64):
    batch_size,num_head,num_q, head_dim = q.shape
    num_kv = k.shape[-2]
    sm_scale = 1.0 / (head_dim ** 0.5)
    out = torch.empty_like(q)
    grid = (batch_size,num_head,triton.cdiv(num_q, BLOCK_M),)
    flash_att_kernel_batched[grid](q, k, v, out, num_q, num_kv, sm_scale,NUM_HEADS=num_head,
                           HEAD_DIM=head_dim, BLOCK_SIZE_Q=BLOCK_M, BLOCK_SIZE_KV=BLOCK_N)
    return out

# q,k,v=torch.randn(1024,64,device="cuda:2"),torch.randn(1024,64,device="cuda:2"),torch.randn(1024,64,device="cuda:2")

q,k,v=torch.randn(2,4,1024,64,device="cuda:2").to(torch.bfloat16).contiguous(),torch.randn(2,4,1024,64,device="cuda:2").to(torch.bfloat16).contiguous(),torch.randn(2,4,1024,64,device="cuda:2").to(torch.bfloat16).contiguous()
ms_triton = triton.testing.do_bench(lambda: flash_att_block(q,k,v))

ms_triton


import torch
import torch.nn.functional as F
import triton
from torch.nn.attention import sdpa_kernel, SDPBackend

DEVICE = "cuda:2"
DTYPE = torch.bfloat16
HEAD_DIM = 128
BATCH_SIZE=12
NUM_HEAD=12

def ref_attention(q, k, v):
    """Single-head reference, 2D in/out, via SDPA's FlashAttention backend."""
    with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
        return F.scaled_dot_product_attention(
         q,k,v  
        )

def bench_one(seq_len):
    q, k, v = (torch.randn(BATCH_SIZE,NUM_HEAD,seq_len, HEAD_DIM, device=DEVICE, dtype=DTYPE) for _ in range(3))

    # --- correctness ---
    out_triton = flash_att_block(q, k, v).float()
    out_ref    = ref_attention(q, k, v).float()
    max_err = (out_triton - out_ref).abs().max().item()
    ok = torch.allclose(out_triton, out_ref, atol=2e-2, rtol=2e-2)

    # --- timing ---
    ms_triton = triton.testing.do_bench(lambda: flash_att_block(q, k, v))
    ms_flash  = triton.testing.do_bench(lambda: ref_attention(q, k, v))

    # --- TFLOP/s (attention FLOPs ≈ 4 * seq^2 * head_dim: two matmuls) ---
    flops = 4 * BATCH_SIZE*NUM_HEAD* seq_len * seq_len * HEAD_DIM
    tflops_triton = flops / (ms_triton * 1e-3) / 1e12
    tflops_flash  = flops / (ms_flash  * 1e-3) / 1e12

    return seq_len, ms_triton, ms_flash, tflops_triton, tflops_flash, max_err, ok

print(f"{'seq':>6} {'triton(ms)':>11} {'flash(ms)':>10} {'tri TFLOPs':>11} "
      f"{'fl TFLOPs':>10} {'ratio':>6} {'maxerr':>8} {'ok':>4}")
for seq in [1024]:
    s, mt, mf, tt, tf, err, ok = bench_one(seq)
    print(f"{s:>6} {mt:>11.4f} {mf:>10.4f} {tt:>11.1f} {tf:>10.1f} "
          f"{mt/mf:>6.2f} {err:>8.4f} {str(ok):>4}")


   seq  triton(ms)  flash(ms)  tri TFLOPs  fl TFLOPs  ratio   maxerr   ok
  1024      1.8388     0.6926        42.0      111.6   2.66   0.0020 True


In [ ]:
@triton.jit
def flash_att_kernel(q_ptr, k_ptr, v_ptr, out_ptr,
                     num_q, num_kv, sm_scale,
                     HEAD_DIM: tl.constexpr,
                     BLOCK_M: tl.constexpr,
                     BLOCK_N: tl.constexpr):
    pid = tl.program_id(0)

    q_bp = tl.make_block_ptr(q_ptr, (num_q, HEAD_DIM), (HEAD_DIM, 1),
                             (pid * BLOCK_M, 0), (BLOCK_M, HEAD_DIM), (1, 0))
    q = tl.load(q_bp)                              # (BLOCK_M, HEAD_DIM)

    m   = tl.full((BLOCK_M,), -float("inf"), tl.float32)
    l   = tl.zeros((BLOCK_M,), tl.float32)
    acc = tl.zeros((BLOCK_M, HEAD_DIM), tl.float32)

    k_bp = tl.make_block_ptr(k_ptr, (num_kv, HEAD_DIM), (HEAD_DIM, 1),
                             (0, 0), (BLOCK_N, HEAD_DIM), (1, 0))
    v_bp = tl.make_block_ptr(v_ptr, (num_kv, HEAD_DIM), (HEAD_DIM, 1),
                             (0, 0), (BLOCK_N, HEAD_DIM), (1, 0))

    for _ in range(0, num_kv, BLOCK_N):
        k = tl.load(k_bp)                          # (BLOCK_N, HEAD_DIM)
        v = tl.load(v_bp)                          # (BLOCK_N, HEAD_DIM)

        s = tl.dot(q, tl.trans(k)) * sm_scale      # (BLOCK_M, BLOCK_N)  <- real matmul

        m_new = tl.maximum(m, tl.max(s, axis=1))   # (BLOCK_M,)
        p     = tl.exp(s - m_new[:, None])         # (BLOCK_M, BLOCK_N)
        corr  = tl.exp(m - m_new)                  # (BLOCK_M,)

        l   = corr * l + tl.sum(p, axis=1)
        acc = corr[:, None] * acc + tl.dot(p, v)   # (BLOCK_M, HEAD_DIM) <- real matmul
        m   = m_new

        k_bp = tl.advance(k_bp, (BLOCK_N, 0))
        v_bp = tl.advance(v_bp, (BLOCK_N, 0))

    acc = acc / l[:, None]

    o_bp = tl.make_block_ptr(out_ptr, (num_q, HEAD_DIM), (HEAD_DIM, 1),
                             (pid * BLOCK_M, 0), (BLOCK_M, HEAD_DIM), (1, 0))
    tl.store(o_bp, acc)


def flash_att(q, k, v, BLOCK_M=64, BLOCK_N=64):
    num_q, head_dim = q.shape
    num_kv = k.shape[0]
    sm_scale = 1.0 / (head_dim ** 0.5)
    out = torch.empty_like(q)
    grid = (triton.cdiv(num_q, BLOCK_M),)
    flash_att_kernel[grid](q, k, v, out, num_q, num_kv, sm_scale,
                           HEAD_DIM=head_dim, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N)
    return out

q,k,v=torch.randn(1024,64,device="cuda:2"),torch.randn(1024,64,device="cuda:2"),torch.randn(1024,64,device="cuda:2")
ms_triton = triton.testing.do_bench(lambda: flash_att(q,k,v))



In [ ]:
q,k,v=torch.randn(1024,64,device="cuda:2"),torch.randn(1024,64,device="cuda:2"),torch.randn(1024,64,device="cuda:2")


q,k,v=q.to(torch.float32),k.to(torch.float32),v.to(torch.float32)
ms_triton = triton.testing.do_bench(lambda: flash_att(q,k,v))
ms_torch  = triton.testing.do_bench(lambda: F.scaled_dot_product_attention(q,k,v))
ms_ours = triton.testing.do_bench(lambda: flash_att_block(q,k,v))

print(torch.cuda.get_device_name())
print(f"triton: {ms_triton:.4f} ms" )
print(f"torch : {ms_torch:.4f} ms")
print(f"ours : {ms_ours:.4f} ms")


NVIDIA RTX A6000
triton: 0.0981 ms
torch : 0.2442 ms
ours : 0.0929 ms


In [ ]:
with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CUDA, torch.profiler.ProfilerActivity.CPU]
) as prof:
    out = flash_att(q, k, v)

# Print the top executing kernel paths
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

---------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                       Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
---------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
           aten::empty_like         0.97%      13.090us         4.77%      64.639us      64.639us             1  
        aten::empty_strided         3.80%      51.549us         3.80%      51.549us      51.549us             1  
    Activity Buffer Request        58.83%     797.171us        58.83%     797.171us     797.171us             1  
           cuLaunchKernelEx        35.55%     481.745us        35.55%     481.745us     481.745us             1  
      cudaDeviceSynchronize         0.85%      11.480us         0.85%      11.480us      11.480us             1  
---------------------------  ------------  ------------  ------------  ------------  ---

In [ ]:
print(q)

tensor([[ 0.1926, -0.4079, -0.3516,  0.4800, -1.2046,  1.8350, -2.1339,  0.0404],
        [ 0.5624, -0.3969,  0.0708, -1.2840,  0.7230, -0.4499,  0.2700, -0.0588]],
       device='cuda:0')


In [ ]:
q[0].shape

torch.Size([8])

In [ ]:
print(q[0],k[0],"Q"*10,q,"K"*10,k,"V"*10,v)

tensor([ 0.1926, -0.4079, -0.3516,  0.4800, -1.2046,  1.8350, -2.1339,  0.0404],
       device='cuda:0') tensor([ 0.1763,  0.2500, -2.1208,  1.3479,  0.1798,  0.9255, -1.9181, -1.0089],
       device='cuda:0') QQQQQQQQQQ tensor([[ 0.1926, -0.4079, -0.3516,  0.4800, -1.2046,  1.8350, -2.1339,  0.0404],
        [ 0.5624, -0.3969,  0.0708, -1.2840,  0.7230, -0.4499,  0.2700, -0.0588]],
       device='cuda:0') KKKKKKKKKK tensor([[ 0.1763,  0.2500, -2.1208,  1.3479,  0.1798,  0.9255, -1.9181, -1.0089],
        [-1.2676,  0.7956, -0.2881, -1.7570,  0.5868,  0.3599, -0.0048, -0.4813]],
       device='cuda:0') VVVVVVVVVV tensor([[ 1.4381,  0.3255, -0.5288,  0.4665, -1.2603, -0.5218, -0.4184,  0.5082],
        [-0.9587,  0.2325,  0.5359,  0.3523,  0.7175, -1.2287,  0.4167, -1.4823]],
       device='cuda:0')
